# PickupAuth — Cuaderno de implementación

Implementa de punta a punta lo que describe `PickupAuth — Cuaderno de implementación.md`:
preparación de datos, etapa 1 (preentrenamiento autosupervisado de las capas bajas), etapa 2
(ramas por fase y fusión con el dataset propio, sin identidad), etapa 3 (aprendizaje del dueño),
las pruebas, las rondas R1–R5, las líneas base de trabajos previos (Secure Pick Up, Please Hold On,
Boshoff) y la comparación con Motion ID.

**Cómo correrlo**

- En Colab: *Entorno de ejecución → Cambiar tipo → GPU*, y ejecutar todo. La celda de configuración monta
  Google Drive y busca los datos en `RAIZ_DRIVE` (por defecto `MyDrive/PickupAuth`).
- En un servidor con Jupyter (por ejemplo, el de la facultad): no hay `drive.mount`. Todo vive en una carpeta
  del servidor (`RAIZ`, por defecto `~/pickupauth`). Los datos se suben a mano en un solo zip,
  `pickupauth_datos.zip`, y la celda 0.1 lo descomprime en `RAIZ` la primera vez; después no se baja nada de
  internet. Para corridas largas conviene ejecutar el cuaderno sin el navegador abierto, ver el final de la celda 0.1.
- `MODO = "FULL"` corre el trabajo completo; `MODO = "SMOKE"` usa pocos datos y pocas épocas para
  comprobar en minutos que todo funciona de punta a punta (así se verificó en la PC local).
- Todo lo que tarda se guarda en `RESULTADOS/` (en Drive): cachés de datos, checkpoints de cada etapa y la
  tabla de seguimiento. Si Colab se desconecta, al volver a ejecutar se retoma donde quedó.

**Qué tiene que haber en Drive** (`MyDrive/PickupAuth/`):

```
PickupAuth/
  dataset_PickAuth/          ← la carpeta que llena la app (S01/, S02/, … con sus zips diarios); o el zip que baja Drive
  datasets/HMOG/             ← los <usuario>.zip de HMOG (tal como están en la PC)
  datasets/HHAR/heterogeneity+activity+recognition.zip
```

En Colab, ExtraSensory (crudo) y Motion ID se leen directamente de internet, y no hace falta cuenta. Motion ID en
campo pesa ~150 GB: el cuaderno baja un par persona-teléfono a la vez y lo borra al terminar. Se puede limitar con
`MID_CAMPO_PARES`.

**Qué trae `pickupauth_datos.zip`** (para el servidor; se descomprime en `RAIZ`):

```
pickupauth/
  dataset_PickAuth/
  datasets/HMOG/  datasets/HHAR/
  datasets/ExtraSensory/ExtraSensory.raw_measurements.raw_acc.zip, …proc_gyro.zip
  datasets/MotionID/<slug>.json y <slug>/…   ← movimiento específico completo y los 12 pares de MID_CAMPO_PARES
```

Si `ES_ACC`, `ES_GYR` o `MID_LOCAL` existen, se leen de ahí; si no, de internet como en Colab.

**Reglas fijas** (del `.md`): solo el dueño en el modelo final; persona, no teléfono; no se pegan tramos de
datasets distintos; solo redes neuronales en nuestro modelo; una variable por prueba, 3 semillas.
Las líneas base de trabajos previos (sección 11) no son redes y algunas usan negativos de otras personas,
como en sus artículos: sirven para comparar, no son el modelo de la tesis.

## 0. Configuración

Lo único que normalmente hay que tocar está en esta celda.

In [ ]:
import os, sys, json, math, time, re, io, zipfile, shutil, random, hashlib, warnings, threading, urllib.request, urllib.parse
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

EN_COLAB = "google.colab" in sys.modules
# "colab" o "servidor" (cualquier Jupyter fuera de Colab, como el de la facultad)
ENTORNO = os.environ.get("PICKUPAUTH_ENTORNO", "colab" if EN_COLAB else "servidor")
MODO = os.environ.get("PICKUPAUTH_MODO", "FULL")          # "FULL" o "SMOKE"
SMOKE = MODO == "SMOKE"

if ENTORNO == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_DRIVE = Path("/content/drive/MyDrive/PickupAuth")
    CACHE_LOCAL = Path("/content/cache")                 # disco rápido de la máquina
else:
    # En el servidor, todo va en una carpeta propia con espacio suficiente (~60 GB con Motion ID en campo).
    # Si tu cuenta tiene poco espacio en el home, apunta RAIZ a un disco de datos (pregunta en la facultad cuál).
    RAIZ_DRIVE = Path(os.environ.get("PICKUPAUTH_RAIZ", "~/pickupauth")).expanduser()
    CACHE_LOCAL = Path(os.environ.get("PICKUPAUTH_CACHE", str(RAIZ_DRIVE / "cache"))).expanduser()
    RAIZ_DRIVE.mkdir(parents=True, exist_ok=True)

CFG = dict(
    # --- dónde están los datos -------------------------------------------------------------
    # Carpeta dataset_PickAuth de Drive (una subcarpeta por participante con sus zips diarios),
    # o el zip que baja Drive al descargar esa carpeta.
    PROPIO=os.environ.get("PICKUPAUTH_PROPIO", str(RAIZ_DRIVE / "dataset_PickAuth")),
    HMOG=os.environ.get("PICKUPAUTH_HMOG", str(RAIZ_DRIVE / "datasets/HMOG")),            # carpeta con <usuario>.zip
    HHAR=os.environ.get("PICKUPAUTH_HHAR", str(RAIZ_DRIVE / "datasets/HHAR/heterogeneity+activity+recognition.zip")),
    RESULTADOS=os.environ.get("PICKUPAUTH_RESULTADOS", str(RAIZ_DRIVE / ("resultados_smoke" if SMOKE else "resultados"))),
    # ExtraSensory crudo (acelerómetro y giroscopio): zips locales si están; si no, se lee por partes desde UCSD
    ES_ACC=os.environ.get("PICKUPAUTH_ES_ACC", str(RAIZ_DRIVE / "datasets/ExtraSensory/ExtraSensory.raw_measurements.raw_acc.zip")),
    ES_GYR=os.environ.get("PICKUPAUTH_ES_GYR", str(RAIZ_DRIVE / "datasets/ExtraSensory/ExtraSensory.raw_measurements.proc_gyro.zip")),
    ES_ACC_URL="http://extrasensory.ucsd.edu/data/raw_measurements/ExtraSensory.raw_measurements.raw_acc.zip",
    ES_GYR_URL="http://extrasensory.ucsd.edu/data/raw_measurements/ExtraSensory.raw_measurements.proc_gyro.zip",
    # Motion ID: copia local (misma estructura que en Kaggle) si está; si no, Kaggle con descarga anónima
    MID_LOCAL=os.environ.get("PICKUPAUTH_MID", str(RAIZ_DRIVE / "datasets/MotionID")),
    MID_URL="https://www.kaggle.com/api/v1/datasets/download/djaarf/{slug}/{path}",
    MID_LISTA="https://www.kaggle.com/api/v1/datasets/list/djaarf/{slug}?pageSize=200",

    # --- formato común -------------------------------------------------------------------
    FS=50, PRE_S=8, POST_S=5,          # ventana del dataset propio: −8 s a +5 s del desbloqueo
    X_MAX=5,                           # segundos de uso máximos (la app guarda +5 s)
    VENTANA=128, SOLAPE=0.5,           # ventanas de la etapa 1 (2,56 s a 50 Hz)
    NORM="zscore",                     # normalización por participante: "zscore" | "escala"

    # --- qué fuentes se usan -------------------------------------------------------------
    USAR_HMOG=True, USAR_ES=True, USAR_HHAR=True,
    USAR_MID_ESPECIFICO=True,          # comparación en condiciones ideales (camino 2)
    USAR_MID_CAMPO=True,               # 6 personas × 6 teléfonos; pesado (~150 GB en total, se procesa por partes)
    # Pares (persona, teléfono) de Motion ID en campo. Cada par son ~2,8 GB (dos semanas continuas de sensores).
    # Por defecto, las 6 personas en los teléfonos 0 y 1: 12 pares, ~34 GB. None = los 36 pares (~100 GB, muchas horas).
    MID_CAMPO_PARES=[(u, t) for u in range(1, 7) for t in (0, 1)],
    MAX_VENT_HMOG=150_000, MAX_VENT_ES=150_000, ES_MIN_POR_USUARIO=300, HHAR_VENT_POR_GRUPO=300, HHAR_FILAS_POR_GRUPO=60_000,

    # --- particiones y dueño -------------------------------------------------------------
    MIN_EPISODIOS=60,                  # mínimo de desbloqueos válidos para entrar al estudio
    PROP_PARTICION=(0.6, 0.2, 0.2),    # entrenamiento, validación, prueba (por persona)
    ENROLAR_N=40, CALIBRAR_N=20,       # enrolamiento y calibración del umbral (en orden temporal)
    SEMANAS_R3=(1,),                   # la ronda 3 enrola con la semana 1 y evalúa con lo que sigue (la recolección dura ~2 semanas)
    DIAS_R3=(1, 3),                    # provisional mientras no hay 2 semanas: enrola con el día 1 y con los días 1–3
    MAX_IMPOSTOR_OTROS=60,             # episodios por persona usados como impostores de otros teléfonos
    FRR_OBJETIVO=0.05, HIBRIDO_SEGUNDO="svdd",      # el híbrido promedia conjunto y este esquema
    ACTUALIZAR_POR_DIA=10,

    # --- modelo y entrenamiento ----------------------------------------------------------
    D=64, DZ=128, PROY=1024, LAMBDA_BT=0.005,
    ARQUITECTURAS=["cnn", "tcn", "gru", "cnn_gru", "transformer", "cnn_transformer"],
    SIMPLICIDAD_OBJ=["barlow", "vicreg", "enmascarado", "entre_sensores"],
    E1_EPOCAS=100, E1_LOTE=512, E1_PASOS_MAX=None,
    E2_EPOCAS=100, E2_LOTE=64, E2_PASOS_POR_EPOCA=60, E2_BAJAS="congelar",   # "congelar" | "lento" | "libre"
    E3_EPOCAS=40, E3_LOTE=32,
    P_DROPOUT_FASE=0.2, PESOS_AUX=dict(origen=0.5, fases=0.5, real=0.5, mascara=1.0),
    SEMILLAS=[0, 1, 2], UMBRAL_EMPATE=1.0,          # puntos de EER
    BOOTSTRAP=1000,
)

if SMOKE:   # lo mínimo para recorrer todo el pipeline en una CPU
    CFG.update(MAX_VENT_HMOG=3000, MAX_VENT_ES=1500, ES_MIN_POR_USUARIO=6, HHAR_VENT_POR_GRUPO=20, HHAR_FILAS_POR_GRUPO=6000,
               MIN_EPISODIOS=30, ENROLAR_N=15, CALIBRAR_N=8, MAX_IMPOSTOR_OTROS=15,
               D=32, DZ=64, PROY=256, E1_EPOCAS=1, E1_LOTE=64, E1_PASOS_MAX=4,
               E2_EPOCAS=2, E2_LOTE=16, E2_PASOS_POR_EPOCA=3, E3_EPOCAS=2, E3_LOTE=8,
               SEMILLAS=[0], BOOTSTRAP=50,
               SMOKE_SUJETOS=10, SMOKE_EP_POR_SUJETO=70, SMOKE_HMOG_USUARIOS=2, SMOKE_HMOG_SESIONES=2,
               SMOKE_ES_USUARIOS=2, SMOKE_MID_ENTRENAR=4, SMOKE_MID_PRUEBA=3)

# GPU: si hay varias, se usa la que tiene más memoria libre (en servidores compartidos otra puede estar ocupada).
# Para elegir una a mano: PICKUPAUTH_GPU=1 (o CUDA_VISIBLE_DEVICES) antes de abrir el cuaderno.
if "CUDA_VISIBLE_DEVICES" not in os.environ:
    if os.environ.get("PICKUPAUTH_GPU"):
        os.environ["CUDA_VISIBLE_DEVICES"] = os.environ["PICKUPAUTH_GPU"]
    elif shutil.which("nvidia-smi"):
        try:
            libres = [int(x) for x in os.popen("nvidia-smi --query-gpu=memory.free --format=csv,noheader,nounits").read().split()]
            if libres:
                os.environ["CUDA_VISIBLE_DEVICES"] = str(max(range(len(libres)), key=lambda i: libres[i]))
                print(f"GPU elegida: {os.environ['CUDA_VISIBLE_DEVICES']} (memoria libre por GPU, MiB: {libres})")
        except Exception as e:
            print("No se pudo consultar nvidia-smi:", e)

RES = Path(CFG["RESULTADOS"]); RES.mkdir(parents=True, exist_ok=True)
CACHE = CACHE_LOCAL; CACHE.mkdir(parents=True, exist_ok=True)
(RES / "ckpt").mkdir(exist_ok=True); (RES / "datos").mkdir(exist_ok=True)
print("MODO", MODO, "| resultados en", RES, "| caché local", CACHE)

In [ ]:
import importlib.util
faltan = [m for m in ("numba", "pyarrow", "sklearn", "scipy", "pandas", "matplotlib") if importlib.util.find_spec(m) is None]
if faltan:
    nombres = {"sklearn": "scikit-learn"}
    os.system(f"{sys.executable} -m pip install -q " + " ".join(nombres.get(m, m) for m in faltan))
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Falta PyTorch. En una terminal del servidor, con el mismo Python del kernel: "
                       f"{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cu124 "
                       "(cu124 sirve para drivers con CUDA 12.4 o más; revisa con nvidia-smi). Después reinicia el kernel.")
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy import signal as sps
from sklearn.metrics import roc_curve
warnings.filterwarnings("ignore", category=UserWarning)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
if not torch.cuda.is_available():
    torch.set_num_threads(max(1, (os.cpu_count() or 2) - 1))
print("torch", torch.__version__, "| dispositivo", DEV)

def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def guardar_json(obj, p): Path(p).write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=str))

## 0.1 Traer los datos (solo en el servidor)

En Colab no hace falta: Drive ya está montado. En el servidor:

- **Todo viene en `pickupauth_datos.zip`**, que se sube a mano al servidor (al directorio principal del entorno,
  `/notebook`, o al home). La celda lo busca en `DATOS_ZIP` y lo descomprime en `RAIZ` sin pisar lo que ya existe:
  si en el servidor hay una versión más nueva de `dataset_PickAuth`, se conserva y solo se agregan los archivos
  que le falten. Al terminar deja la marca `RAIZ/.datos_descomprimidos` y no vuelve a descomprimir. Hacen falta ~45 GB libres además
  del zip; con `BORRAR_ZIP=True` se borra el zip al terminar para recuperar ~29 GB.
- **Opcional: traer los días nuevos del dataset propio desde Drive con rclone** (`PICKUPAUTH_SINCRONIZAR=1`).
  rclone copia carpetas de Google Drive sin montar nada; la celda lo descarga si no está. Hay que autorizarlo
  **una sola vez**, en una terminal del servidor (en Jupyter: *File → New → Terminal*):

  ```
  ~/pickupauth/bin/rclone config
  ```
  `n` (nuevo remoto) → nombre `gdrive` → tipo `drive` → Enter en *client_id* y *client_secret* → `1` (acceso
  completo) → Enter → `n` en *Edit advanced config* → **`n` en *Use web browser to automatically authenticate*** (el
  servidor no tiene navegador). rclone muestra un comando `rclone authorize "drive" "..."`: córrelo en tu PC (con
  rclone instalado allí, https://rclone.org/downloads/), autoriza con tu cuenta de Google en el navegador y pega
  en el servidor el texto que devuelve. Después: `n` (no es Shared Drive) → `y` → `q`.

**Corridas largas.** Si cierras el navegador, el kernel de Jupyter sigue corriendo pero la salida de las celdas
se pierde. Para el modo FULL conviene ejecutarlo desde una terminal del servidor, dentro de `tmux` o con `nohup`:

```
cd /notebook && nohup jupyter nbconvert --to notebook --execute PickupAuth_implementacion.ipynb \
    --output salida_full.ipynb --ExecutePreprocessor.timeout=-1 > corrida.log 2>&1 &
```
Todo queda guardado en `RESULTADOS/` a medida que avanza; si se corta, se vuelve a lanzar y retoma.

In [ ]:
CFG.update(
    # Zip con todos los datos, subido a mano. Se usa el primero que exista.
    DATOS_ZIP=[os.environ.get("PICKUPAUTH_ZIP", ""), "/notebook/pickupauth_datos.zip", "pickupauth_datos.zip",
               str(Path("~/pickupauth_datos.zip").expanduser()), str(RAIZ_DRIVE / "pickupauth_datos.zip")],
    BORRAR_ZIP=False,                       # borrar el zip después de descomprimirlo
    SINCRONIZAR_DRIVE=os.environ.get("PICKUPAUTH_SINCRONIZAR", "0") == "1",   # traer días nuevos del dataset propio con rclone
    REMOTO_DRIVE="gdrive",                  # nombre del remoto de rclone
    # Dónde está cada cosa en Drive (rutas dentro del remoto). La app sube los zips diarios a dataset_PickAuth.
    DRIVE_PROPIO="dataset_PickAuth",
    DRIVE_HMOG="PickUpAuth/HMOG",
    DRIVE_HHAR="PickUpAuth/HHAR",           # si no está, se baja de UCI
    DRIVE_RESULTADOS="PickUpAuth/resultados",            # trae particiones.json para no cambiar las particiones
    DRIVE_SUBIR_RESULTADOS="PickUpAuth/resultados_servidor",  # adonde se suben los resultados al terminar
    HHAR_URL="https://archive.ics.uci.edu/static/public/344/heterogeneity+activity+recognition.zip",
)

def bajar(url, destino):
    """Descarga con un User-Agent de navegador (algunos servidores rechazan el de Python)."""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=600) as r, open(destino, "wb") as f: shutil.copyfileobj(r, f, 1 << 20)

def rclone_bin():
    r = shutil.which("rclone")
    if r: return r
    destino = RAIZ_DRIVE / "bin" / "rclone"
    if not destino.exists():
        print("Descargando rclone…")
        destino.parent.mkdir(parents=True, exist_ok=True)
        z = RAIZ_DRIVE / "bin" / "rclone.zip"
        bajar("https://downloads.rclone.org/rclone-current-linux-amd64.zip", z)
        with zipfile.ZipFile(z) as zz:
            m = [n for n in zz.namelist() if n.endswith("/rclone")][0]
            destino.write_bytes(zz.read(m))
        destino.chmod(0o755); z.unlink()
    return str(destino)

def remoto_ok():
    rc = rclone_bin()
    if os.system(f'"{rc}" listremotes | grep -q "^{CFG["REMOTO_DRIVE"]}:$"') != 0:
        raise RuntimeError(f"rclone no tiene configurado el remoto '{CFG['REMOTO_DRIVE']}'. "
                           f"Autorízalo una vez en una terminal: {rc} config (instrucciones arriba).")
    return rc

def sincronizar(origen, destino, obligatorio=True):
    """Copia (solo lo nuevo) una carpeta de Drive al servidor."""
    rc = remoto_ok()
    Path(destino).mkdir(parents=True, exist_ok=True)
    cmd = f'"{rc}" copy "{CFG["REMOTO_DRIVE"]}:{origen}" "{destino}" --transfers 8 --stats-one-line --stats 30s'
    print(cmd); ok = os.system(cmd) == 0
    if obligatorio: assert ok, f"Falló la copia de {origen}"
    return ok

def subir_resultados():
    """Sube RESULTADOS a Drive (solo lo nuevo). Se llama al final del cuaderno."""
    if ENTORNO != "servidor" or not CFG["SINCRONIZAR_DRIVE"]: return
    rc = remoto_ok()
    os.system(f'"{rc}" copy "{CFG["RESULTADOS"]}" "{CFG["REMOTO_DRIVE"]}:{CFG["DRIVE_SUBIR_RESULTADOS"]}" --transfers 8')

def descomprimir_datos():
    """Descomprime pickupauth_datos.zip en RAIZ sin pisar archivos que ya existen (como unzip -n)."""
    marca = RAIZ_DRIVE / ".datos_descomprimidos"          # se escribe solo si la descompresión terminó entera
    if marca.exists(): return
    zp = next((Path(p).expanduser() for p in CFG["DATOS_ZIP"] if p and Path(p).expanduser().is_file()), None)
    if zp is None:
        print("No encontré pickupauth_datos.zip en", [p for p in CFG["DATOS_ZIP"] if p], "— faltan datos locales.")
        return
    with zipfile.ZipFile(zp) as z:
        miembros = [i for i in z.infolist() if not i.is_dir()]
        total = sum(i.file_size for i in miembros); hecho = nuevos = 0; t0 = time.time()
        print(f"Descomprimiendo {zp} ({total / 1e9:.1f} GB) en {RAIZ_DRIVE}…")
        for k, i in enumerate(miembros):
            rel = i.filename.split("/", 1)[1] if i.filename.startswith("pickupauth/") else i.filename
            destino = RAIZ_DRIVE / rel
            hecho += i.file_size
            if destino.exists(): continue
            destino.parent.mkdir(parents=True, exist_ok=True)
            tmp = destino.with_name(destino.name + ".parte")
            with z.open(i) as f, open(tmp, "wb") as g: shutil.copyfileobj(f, g, 1 << 22)
            tmp.rename(destino); nuevos += 1
            if (k + 1) % 50 == 0: print(f"  {hecho / 1e9:.1f}/{total / 1e9:.1f} GB ({time.time() - t0:.0f} s)")
    marca.write_text(str(zp))
    print(f"Listo: {nuevos} archivos nuevos ({time.time() - t0:.0f} s).")
    if CFG["BORRAR_ZIP"]: zp.unlink(); print("Zip borrado.")

if ENTORNO == "servidor":
    descomprimir_datos()
    if CFG["SINCRONIZAR_DRIVE"]:
        sincronizar(CFG["DRIVE_PROPIO"], CFG["PROPIO"])
        if not (Path(CFG["RESULTADOS"]) / "particiones.json").exists():
            sincronizar(CFG["DRIVE_RESULTADOS"], CFG["RESULTADOS"], obligatorio=False)
    if CFG["USAR_HHAR"] and not Path(CFG["HHAR"]).exists():
        print("Descargando HHAR desde UCI…")
        Path(CFG["HHAR"]).parent.mkdir(parents=True, exist_ok=True)
        bajar(CFG["HHAR_URL"], CFG["HHAR"])
    for k in ("PROPIO", "HMOG", "HHAR", "ES_ACC", "ES_GYR", "MID_LOCAL"):
        print(f"{k:9s}", "OK " if Path(CFG[k]).exists() else "FALTA", CFG[k])
    import torch as _t
    print("GPU:", _t.cuda.get_device_name(0) if _t.cuda.is_available() else "no hay GPU visible (revisa CUDA_VISIBLE_DEVICES)")

## 1. Señal: remuestreo, calibración con el reposo y canales comunes

Formato común: 50 Hz, acelerómetro (en g) y giroscopio (rad/s) en los ejes del teléfono, gravedad estimada con un
pasa bajos como canal aparte, y el magnetómetro solo como magnitud relativa a la del reposo. Dos canales más
indican si hay giroscopio y magnetómetro (S14 y S15 no tienen giroscopio; HHAR y ExtraSensory no traen
magnetómetro). Con los datos de entrada en unidades físicas se aplican los aumentos y después la normalización
por participante, que se hace en la GPU para cada lote.

**Calibración con el reposo.** Antes de la toma el teléfono está quieto: el segundo más quieto antes del inicio de
la toma da el sesgo del giroscopio, la ganancia del acelerómetro (|a| debe ser 1 g) y la magnitud de referencia del
magnetómetro. Si no hay reposo, se usa la mediana del episodio (o de la sesión, en los públicos). Es la principal
defensa contra la huella del teléfono.

In [ ]:
G0 = 9.80665
C_DATOS = 10          # acc(3) gyr(3) gravedad(3) |mag| relativa(1)
C_ENT = 12            # + bandera de giroscopio + bandera de magnetómetro
ORIGENES = ["SURFACE", "HAND", "POCKET_OR_BAG", "IN_PLACE", "UNKNOWN"]
ORIGEN_ID = {o: i for i, o in enumerate(ORIGENES)}
FASES = {"pre": 0, "toma": 1, "asentamiento": 2, "uso": 3}

def a_rejilla(t, v, tg, max_hueco=0.2):
    """Interpolación lineal de (t, v) en la rejilla tg; NaN fuera de rango o en huecos > max_hueco s."""
    v = np.asarray(v, np.float64).reshape(len(t), -1)
    out = np.full((len(tg), v.shape[1]), np.nan, np.float32)
    if len(t) < 2: return out
    t, idx = np.unique(np.asarray(t, np.float64), return_index=True); v = v[idx]
    if len(t) < 2: return out
    for k in range(v.shape[1]): out[:, k] = np.interp(tg, t, v[:, k])
    pos = np.searchsorted(t, tg)
    izq = np.clip(pos - 1, 0, len(t) - 1); der = np.clip(pos, 0, len(t) - 1)
    malo = (tg < t[0]) | (tg > t[-1]) | ((t[der] - t[izq]) > max_hueco)
    out[malo] = np.nan
    return out

def remuestrear(t, v, tg, fs_obj, max_hueco=0.2):
    """Lleva una señal irregular a la rejilla tg (fs_obj Hz) con antialias si la fuente es más rápida."""
    t = np.asarray(t, np.float64); v = np.asarray(v, np.float64).reshape(len(t), -1)
    t, iu = np.unique(t, return_index=True); v = v[iu]
    if len(t) < 8: return np.full((len(tg), v.shape[1]), np.nan, np.float32)
    dt = float(np.median(np.diff(t)))
    if not np.isfinite(dt) or dt <= 0: return a_rejilla(t, v, tg, max_hueco)
    fs_src = 1.0 / dt
    if fs_src <= 1.25 * fs_obj:
        return a_rejilla(t, v, tg, max_hueco)
    tu = np.arange(t[0], t[-1] + 1e-9, dt)
    vu = a_rejilla(t, v, tu, max_hueco)
    ok = ~np.isnan(vu).any(1)
    if ok.sum() < 30: return a_rejilla(t, v, tg, max_hueco)
    idx = np.arange(len(tu))
    for k in range(vu.shape[1]): vu[~ok, k] = np.interp(idx[~ok], idx[ok], vu[ok, k])
    b, a = sps.butter(4, min(0.45 * fs_obj / (fs_src / 2), 0.95))
    vf = sps.filtfilt(b, a, vu, axis=0) if len(tu) > 30 else vu
    return a_rejilla(tu[ok], vf[ok], tg, max_hueco)

def rellenar(v):
    """Rellena NaN por interpolación (para filtrar); devuelve copia."""
    v = v.copy(); idx = np.arange(len(v))
    for k in range(v.shape[1]):
        ok = ~np.isnan(v[:, k])
        if ok.sum() == 0: v[:, k] = 0
        elif (~ok).any(): v[~ok, k] = np.interp(idx[~ok], idx[ok], v[ok, k])
    return v

_B_GRAV = {}
def gravedad(acc, fs):
    if fs not in _B_GRAV: _B_GRAV[fs] = sps.butter(2, 0.3 / (fs / 2))
    b, a = _B_GRAV[fs]
    x = rellenar(acc)
    return sps.filtfilt(b, a, x, axis=0).astype(np.float32) if len(x) > 12 else x.astype(np.float32)

def calibrar(acc_g, gyr, mag, fin_reposo, fs):
    """Calibración por episodio. acc_g en g; devuelve acc, gyr, |mag| relativa y si hubo reposo."""
    amag = np.linalg.norm(acc_g, axis=1)
    w = int(fs); rep = None
    if fin_reposo is not None and fin_reposo >= w:
        a = amag[:fin_reposo]
        if not np.isnan(a).any() or np.isnan(a).mean() < 0.2:
            a = rellenar(a[:, None])[:, 0]
            cs = np.cumsum(np.r_[0, a]); cs2 = np.cumsum(np.r_[0, a * a])
            m = (cs[w:] - cs[:-w]) / w; sd = np.sqrt(np.maximum((cs2[w:] - cs2[:-w]) / w - m * m, 0))
            i = int(np.argmin(sd))
            quieto = sd[i] < 0.02
            if gyr is not None and quieto:
                gn = np.linalg.norm(gyr[i:i + w], axis=1)
                quieto = bool(np.isfinite(gn).any()) and np.nanmean(gn) < 0.15
            if quieto: rep = slice(i, i + w)
    if rep is not None:
        ganancia = 1.0 / max(np.nanmean(amag[rep]), 1e-3)
        sesgo = np.nanmean(gyr[rep], 0) if gyr is not None else None
        mref = np.nanmedian(np.linalg.norm(mag[rep], axis=1)) if mag is not None else None
    else:
        ganancia = 1.0 / max(np.nanmedian(amag), 1e-3)
        sesgo = np.nanmedian(gyr, 0) if gyr is not None else None
        mref = np.nanmedian(np.linalg.norm(mag, axis=1)) if mag is not None else None
    ganancia = float(np.clip(ganancia, 0.9, 1.1))
    acc_c = acc_g * ganancia
    gyr_c = gyr - sesgo if gyr is not None else None
    mrel = None
    if mag is not None and mref is not None and np.isfinite(mref) and mref > 1e-3:
        mrel = np.clip(np.linalg.norm(mag, axis=1) / mref - 1.0, -3, 3)
    return acc_c, gyr_c, mrel, rep is not None

def canales(acc_c, gyr_c, mrel, fs):
    """Arma (C_DATOS, T) float32 con ceros donde no hay dato, y las banderas (gyr_ok, mag_ok)."""
    T = len(acc_c)
    X = np.zeros((C_DATOS, T), np.float32)
    X[0:3] = np.nan_to_num(acc_c.T)
    X[6:9] = gravedad(acc_c, fs).T
    gyr_ok = gyr_c is not None and np.isfinite(gyr_c).mean() > 0.9
    mag_ok = mrel is not None and np.isfinite(mrel).mean() > 0.9
    if gyr_ok: X[3:6] = np.nan_to_num(gyr_c.T)
    if mag_ok: X[9] = np.nan_to_num(mrel)
    return X, (float(gyr_ok), float(mag_ok))

def inicio_movimiento(acc_g, hasta, fs, umbral=0.03):
    """Inicio de la toma offline: retrocede desde `hasta` hasta el último segundo quieto (std |a| < umbral)."""
    if hasta is None or hasta < fs: return None
    amag = rellenar(np.linalg.norm(acc_g[:hasta], axis=1)[:, None])[:, 0]
    w = fs // 2
    s = pd.Series(amag).rolling(w).std().values
    quietos = np.where(s < umbral)[0]
    if len(quietos) == 0: return None
    return int(quietos[-1])          # fin de la última media ventana quieta

def etiquetas_fase(T, i_toma, i_on, i_up, fs, x_s):
    """0 pre, 1 toma, 2 asentamiento y desbloqueo, 3 uso, -1 fuera de la ventana usada."""
    f = np.full(T, -1, np.int8)
    fin_toma = min(i_on, i_up)
    f[:max(i_toma, 0)] = 0
    f[max(i_toma, 0):fin_toma] = 1
    f[fin_toma:i_up] = 2
    f[i_up:min(T, i_up + int(round(x_s * fs)))] = 3
    return f

def armar_episodio(t_acc, acc, t_gyr, gyr, t_mag, mag, t_up, t_on, t_toma, fs, pre_s, post_s,
                   valido_desde=None, valido_hasta=None, unidades_acc="ms2"):
    """Episodio en la rejilla común; tiempos en segundos (mismo reloj). Devuelve dict o (None, motivo)."""
    T = int(round((pre_s + post_s) * fs)); i_up = int(round(pre_s * fs))
    tg = t_up + (np.arange(T) - i_up) / fs
    A = remuestrear(t_acc, acc, tg, fs)
    if unidades_acc == "ms2": A = A / G0
    val = ~np.isnan(A).any(1)
    permitido = np.ones(T, bool)
    if valido_desde is not None: permitido &= tg >= valido_desde
    if valido_hasta is not None: permitido &= tg <= valido_hasta
    val &= permitido
    A[~val] = np.nan
    if val[permitido].mean() < 0.8 or val[:i_up].sum() < fs or (val[i_up:].sum() < 0.3 * fs and post_s > 0):
        return None, "cobertura_acc"
    Gy = remuestrear(t_gyr, gyr, tg, fs) if t_gyr is not None and len(t_gyr) > 8 else None
    Mg = remuestrear(t_mag, mag, tg, fs) if t_mag is not None and len(t_mag) > 8 else None
    if Gy is not None: Gy[~val] = np.nan
    if Mg is not None: Mg[~val] = np.nan
    i_on = int(round(pre_s * fs + (t_on - t_up) * fs)) if t_on is not None else i_up
    i_on = int(np.clip(i_on, 0, i_up))
    detectada = t_toma is not None
    if detectada:
        i_toma = int(round(pre_s * fs + (t_toma - t_up) * fs)) - fs      # inicio de la toma − 1 s
    else:
        i0 = inicio_movimiento(A, min(i_on, i_up), fs)
        i_toma = (i0 - fs) if i0 is not None else min(i_on, i_up) - 3 * fs
    i_toma = int(np.clip(i_toma, 0, min(i_on, i_up)))
    primer_valido = int(np.argmax(val)) if val.any() else 0
    i_toma = max(i_toma, primer_valido)
    acc_c, gyr_c, mrel, reposo = calibrar(A, Gy, Mg, i_toma if i_toma >= fs else None, fs)
    X, banderas = canales(acc_c, gyr_c, mrel, fs)
    X[:, ~val] = 0
    fases = etiquetas_fase(T, i_toma, i_on, i_up, fs, post_s)
    fases[~val] = -1
    return dict(X=X, val=val, fases=fases, banderas=banderas, reposo=reposo, toma_detectada=detectada,
                i_toma=i_toma, i_on=i_on, i_up=i_up), None

def ventanas(X, val, largo, solape):
    """Corta (C, T) en ventanas de `largo` con solape; descarta las que tienen huecos."""
    paso = max(1, int(largo * (1 - solape))); out = []
    for s in range(0, X.shape[1] - largo + 1, paso):
        if val[s:s + largo].all(): out.append(X[:, s:s + largo])
    return out

## 2. Dataset propio

Lee la carpeta `dataset_PickAuth` de Drive (o el zip que entrega Drive al descargarla), junta los zips diarios
de cada participante como `tools/collect_drive.py` y arma cada episodio con `armar_episodio`.

- **Anclas.** `USER_PRESENT` es el ancla (`meta.json`, y si falta, `events.csv`). La toma empieza en
  `t_motion_onset_ns` o `t_transition_ns`; si la app no la detectó (episodios `in_place` o tomas que se le
  escaparon), se estima offline retrocediendo hasta el último reposo.
- **Tramos**: toma (inicio − 1 s → pantalla encendida), asentamiento y desbloqueo (pantalla → `USER_PRESENT`),
  uso (`USER_PRESENT` → +x s).
- **Falsos disparos**: se alinean con la transición en el segundo 3 y solo se usan en la tarea auxiliar
  real/falso de la etapa 2, nunca al evaluar.
- **Control de calidad**: se descartan episodios sin acelerómetro suficiente o con anclas incoherentes; se anota
  cuántos y por qué.

In [ ]:
def recolectar_propio(fuente, destino):
    """Junta <participante>/<participante>_<fecha>.zip en destino/<participante>/episodes/..."""
    fuente = Path(fuente)
    if fuente.suffix == ".zip":
        tmp = CACHE / "drive_zip"
        if not tmp.exists():
            with zipfile.ZipFile(fuente) as z: z.extractall(tmp)
        fuente = tmp
    carpetas = [p for p in fuente.rglob("*") if p.is_dir() and re.fullmatch(r"S\d+", p.name) and any(p.glob("*.zip"))]
    for subj in sorted(carpetas, key=lambda p: p.name):
        if subj.name == "S00": continue
        dest = Path(destino) / subj.name
        for zp in sorted(subj.glob("*.zip")):
            marca = dest / "aplicados" / (zp.name + ".ok")
            if marca.exists(): continue
            with zipfile.ZipFile(zp) as z:
                for info in z.infolist():
                    if info.is_dir() or not info.filename.startswith("episodes/"): continue
                    tgt = dest / info.filename; tgt.parent.mkdir(parents=True, exist_ok=True)
                    tgt.write_bytes(z.read(info))
            marca.parent.mkdir(parents=True, exist_ok=True); marca.touch()
    return sorted(p.name for p in Path(destino).iterdir() if p.is_dir()) if Path(destino).exists() else []

def _leer_csv(p):
    if not p.exists() or p.stat().st_size < 20: return None, None
    d = pd.read_csv(p)
    if len(d) < 2: return None, None
    return d["t_ns"].to_numpy(np.int64), d[["x", "y", "z"]].to_numpy(np.float64)

def _evento(ev, nombre, antes_de=None):
    if ev is None: return None
    t = ev.loc[ev["name"] == nombre, "t_ns"].to_numpy(np.int64)
    if antes_de is not None: t = t[t <= antes_de]
    return int(t[-1]) if len(t) else None

def leer_episodio_propio(d, fs, post_s=None):
    d = Path(d)
    try: m = json.loads((d / "meta.json").read_text())
    except Exception: return None, "meta_ilegible"
    kind = m.get("kind") or d.name.split("_")[0]
    if kind == "false":  kind = "false_trigger"
    ev = pd.read_csv(d / "events.csv") if (d / "events.csv").exists() else None
    ta, a = _leer_csv(d / "acc.csv")
    if ta is None: return None, "sin_acc"
    tg_, g = _leer_csv(d / "gyr.csv"); tm, mg = _leer_csv(d / "mag.csv")
    pre = m.get("config", {}).get("pre_s", 8); post = m.get("config", {}).get("post_s", 5)
    s = lambda t: None if t is None else t / 1e9
    fecha = pd.to_datetime("_".join(d.name.split("_")[-3:-1]), format="%Y%m%d_%H%M%S", errors="coerce")
    base = dict(sujeto=m.get("subject") or d.parent.parent.name, tipo=kind, dir=d.name, fecha=fecha,
                origen=m.get("origin_detected") or "UNKNOWN", impostor=bool(m.get("impostor_session", False)),
                modelo=(m.get("device") or {}).get("model"), version=m.get("app_version", "0.2"),
                fs_real=float((len(ta) - 1) / max((ta[-1] - ta[0]) / 1e9, 1e-3)),
                pantalla_a_desbloqueo_ms=float(m.get("screen_to_unlock_ms") or np.nan))
    if kind == "false_trigger":
        tt = m.get("t_transition_ns") or m.get("t_motion_onset_ns")
        if not tt: return None, "falso_sin_transicion"
        # transición en el segundo 3 de una ventana de 9 s (la app guarda −3 s a +6 s)
        r, mot = armar_episodio(s(ta), a, s(tg_), g, s(tm), mg, t_up=s(tt) + 3.0, t_on=None, t_toma=s(tt),
                                fs=fs, pre_s=CFG["PRE_S"], post_s=CFG["POST_S"],
                                valido_desde=s(m["window_from_ns"]), valido_hasta=s(m["window_to_ns"]))
        if r is None: return None, "falso_" + mot
        r["fases"][:] = np.where(r["val"], -1, -1)     # sin fases: solo la tarea real/falso
        r["i_toma"] = int(round((CFG["PRE_S"] - 3.0) * fs)) - fs
        return {**base, **r}, None
    t_up = m.get("t_user_present_ns") or _evento(ev, "user_present")
    if not t_up: t_up = m["window_to_ns"] - int(post * 1e9)
    t_on = m.get("t_screen_on_ns") or _evento(ev, "screen_on", antes_de=t_up + int(2e9))
    t_toma = m.get("t_motion_onset_ns") or m.get("t_transition_ns") or None
    if kind == "in_place": t_toma = None; base["origen"] = "IN_PLACE"
    if t_on is not None and (t_up - t_on) > 30e9: t_on = None
    if t_toma is not None and not (t_up - 30e9 < t_toma < t_up): t_toma = None
    r, mot = armar_episodio(s(ta), a, s(tg_), g, s(tm), mg, t_up=s(t_up), t_on=s(t_on), t_toma=s(t_toma), fs=fs,
                            pre_s=CFG["PRE_S"], post_s=CFG["POST_S"],
                            valido_desde=s(m["window_from_ns"]), valido_hasta=s(m["window_to_ns"]))
    if r is None: return None, mot
    return {**base, **r}, None

def _leer_un_dir(args):
    d, fs = args
    try:
        return leer_episodio_propio(d, fs)
    except Exception as e:
        return None, "error:" + type(e).__name__

def cargar_propio(fs=None, forzar=False):
    """Devuelve (meta DataFrame, X float32 (N,C_DATOS,T), val bool, fases int8, banderas (N,2)). Usa caché."""
    fs = fs or CFG["FS"]
    central = CACHE / "propio_central"
    sujetos = recolectar_propio(CFG["PROPIO"], central)
    dirs = sorted(str(p) for s in sujetos for p in (central / s / "episodes").glob("*") if p.is_dir())
    if SMOKE:   # pocos sujetos y episodios, para probar
        por = defaultdict(list)
        for p in dirs: por[Path(p).parent.parent.name].append(p)
        cand = sorted(por, key=lambda s: -sum("unlock_" in Path(p).name for p in por[s]))[:CFG["SMOKE_SUJETOS"]]
        dirs = []
        for s in cand:
            u = [p for p in por[s] if "unlock_" in Path(p).name][:CFG["SMOKE_EP_POR_SUJETO"]]
            f = [p for p in por[s] if "false_trigger_" in Path(p).name][:10]
            dirs += u + f
    huella = hashlib.md5(("|".join(dirs) + str(fs) + "v2").encode()).hexdigest()[:10]   # cambiar "v2" si cambia el preprocesamiento
    cp = CACHE / f"propio_{fs}hz_{huella}.npz"
    if cp.exists() and not forzar:
        z = np.load(cp, allow_pickle=False)
        meta = pd.read_parquet(str(cp).replace(".npz", ".parquet"))
        return meta, z["X"], z["val"], z["fases"], z["banderas"]
    print(f"Leyendo {len(dirs)} episodios propios a {fs} Hz…")
    filas, Xs, Vs, Fs_, Bs, motivos = [], [], [], [], [], Counter()
    from multiprocessing import Pool
    with Pool(max(1, (os.cpu_count() or 2))) as pool:
        for r, mot in pool.imap(_leer_un_dir, [(d, fs) for d in dirs], chunksize=8):
            if r is None: motivos[mot] += 1; continue
            Xs.append(r.pop("X")); Vs.append(r.pop("val")); Fs_.append(r.pop("fases")); Bs.append(r.pop("banderas"))
            filas.append(r)
    meta = pd.DataFrame(filas)
    X = np.stack(Xs); V = np.stack(Vs); FA = np.stack(Fs_); B = np.asarray(Bs, np.float32)
    print("Descartados:", dict(motivos))
    np.savez(cp, X=X, val=V, fases=FA, banderas=B)
    meta.to_parquet(str(cp).replace(".npz", ".parquet"))
    guardar_json(dict(motivos), RES / "datos" / f"descartes_propio_{fs}hz.json")
    return meta, X, V, FA, B

META, XP, VP, FP, BP = cargar_propio()
META["sujeto"] = META["sujeto"].astype(str)
print(META.groupby(["sujeto", "tipo"]).size().unstack(fill_value=0))

### Revisión de un episodio
Comprueba que los tramos caen donde deben: la toma termina en la pantalla encendida y el uso empieza en el
desbloqueo (línea vertical).

In [ ]:
import matplotlib
try:
    EN_JUPYTER = get_ipython().__class__.__name__ == "ZMQInteractiveShell"
except NameError:
    EN_JUPYTER = False
if not EN_JUPYTER: matplotlib.use("Agg")
import matplotlib.pyplot as plt
def graficar_episodio(i, archivo=None):
    X, f, r = XP[i], FP[i], META.iloc[i]
    t = (np.arange(X.shape[1]) - CFG["PRE_S"] * CFG["FS"]) / CFG["FS"]
    fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    ax[0].plot(t, X[0:3].T); ax[0].set_ylabel("acc (g)")
    ax[1].plot(t, X[3:6].T); ax[1].set_ylabel("gyr (rad/s)")
    ax[2].plot(t, f); ax[2].set_ylabel("fase"); ax[2].set_xlabel("s desde USER_PRESENT")
    for a in ax: a.axvline(0, color="k", lw=0.8)
    fig.suptitle(f"{r.sujeto} {r.dir} origen={r.origen} toma_detectada={r.toma_detectada} reposo={r.reposo}")
    fig.tight_layout(); fig.savefig(archivo or RES / "datos" / "episodio_ejemplo.png", dpi=80); plt.close(fig)
_i = int(np.where(META.tipo == "unlock")[0][0]); graficar_episodio(_i)
u = META[META.tipo.isin(["unlock", "in_place"])]
calidad = u.assign(sin_giroscopio=BP[u.index, 0] == 0).groupby("sujeto").agg(
    episodios=("dir", "size"), modelo=("modelo", "first"), fs_real_mediana=("fs_real", "median"),
    toma_detectada=("toma_detectada", "mean"), reposo=("reposo", "mean"), sin_giroscopio=("sin_giroscopio", "mean"))
calidad.to_csv(RES / "datos" / "calidad_por_participante.csv"); print(calidad.round(2).to_string())
print("Desbloqueos válidos:", len(u), "| con toma detectada:", round(u.toma_detectada.mean(), 3),
      "| con reposo para calibrar:", round(u.reposo.mean(), 3))

## 3. Datos públicos para la etapa 1 y la prueba de fuga

- **HMOG** (Drive): sesiones de lectura, escritura y mapa, sentado y caminando. Se toman solo los tramos dentro
  de una tarea (`Activity.csv`), que es cuando el teléfono está en la mano.
- **ExtraSensory** crudo: el acelerómetro (`raw_acc`) y el giroscopio (`proc_gyro`) se leen de los zips locales
  (`ES_ACC`, `ES_GYR`, vienen en `pickupauth_datos.zip`); si no están, por partes desde el servidor de UCSD con
  peticiones por rango (no se bajan los 16 GB).
  Cada archivo es un registro de ~20 s a 40 Hz por minuto.
- **HHAR** (Drive): el mismo movimiento en 4 modelos de teléfono; solo para la prueba de fuga, no se entrena con él.

Todas las ventanas se calibran con la mediana de su sesión (no hay reposo marcado) y se descartan las etiquetas
de actividad y de persona: solo se guarda la sesión, que sirve para normalizar.

In [ ]:
class ArchivoHTTP(io.RawIOBase):
    """Archivo remoto de solo lectura con peticiones por rango, para abrir zips grandes sin bajarlos."""
    def __init__(s, url):
        s.url, s.pos = url, 0
        r = urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=60)
        s.size = int(r.headers["Content-Length"])
    def seekable(s): return True
    def readable(s): return True
    def tell(s): return s.pos
    def seek(s, o, w=0):
        s.pos = {0: o, 1: s.pos + o, 2: s.size + o}[w]; return s.pos
    def readinto(s, b):
        n = min(len(b), s.size - s.pos)
        if n <= 0: return 0
        for intento in range(5):
            try:
                req = urllib.request.Request(s.url, headers={"Range": f"bytes={s.pos}-{s.pos + n - 1}"})
                d = urllib.request.urlopen(req, timeout=60).read(); break
            except Exception:
                if intento == 4: raise
                time.sleep(2 ** intento)
        b[:len(d)] = d; s.pos += len(d); return len(d)

def zip_remoto(url): return zipfile.ZipFile(io.BufferedReader(ArchivoHTTP(url), buffer_size=1 << 16))

def miembro_remoto(url, info):
    """Lee un archivo de un zip remoto con una sola petición por rango (cabecera local + datos)."""
    import zlib, struct
    ini = info.header_offset; fin = ini + 30 + len(info.orig_filename.encode()) + 512 + info.compress_size
    for intento in range(5):
        try:
            req = urllib.request.Request(url, headers={"Range": f"bytes={ini}-{fin}"})
            d = urllib.request.urlopen(req, timeout=60).read(); break
        except Exception:
            if intento == 4: raise
            time.sleep(2 ** intento)
    n_nom, n_extra = struct.unpack("<HH", d[26:30]); a = 30 + n_nom + n_extra
    datos = d[a:a + info.compress_size]
    if info.compress_type == zipfile.ZIP_STORED: return datos
    return zlib.decompressobj(-15).decompress(datos)

def miembro_local(f, info):
    """Lee un archivo de un zip local ya abierto (f), igual que miembro_remoto pero desde el disco."""
    import zlib, struct
    f.seek(info.header_offset); cab = f.read(30)
    n_nom, n_extra = struct.unpack("<HH", cab[26:30]); f.seek(n_nom + n_extra, 1)
    datos = f.read(info.compress_size)
    if info.compress_type == zipfile.ZIP_STORED: return datos
    return zlib.decompressobj(-15).decompress(datos)

def calibrar_sesion(acc_g, gyr, mag):
    """Calibración sin reposo: mediana de la sesión."""
    g = 1.0 / max(np.nanmedian(np.linalg.norm(acc_g, axis=1)), 1e-3)
    acc_c = acc_g * float(np.clip(g, 0.9, 1.1))
    gyr_c = gyr - np.nanmedian(gyr, 0) if gyr is not None else None
    mrel = None
    if mag is not None:
        mm = np.linalg.norm(mag, axis=1); ref = np.nanmedian(mm)
        if np.isfinite(ref) and ref > 1e-3: mrel = np.clip(mm / ref - 1, -3, 3)
    return acc_c, gyr_c, mrel

def ventanas_de_tramo(t_acc, acc, t_gyr, gyr, t_mag, mag, fs, unidades="ms2"):
    """Remuestrea un tramo continuo, lo calibra con su mediana y lo corta en ventanas."""
    t0, t1 = max(t_acc[0], t_gyr[0] if t_gyr is not None else -1e18), min(t_acc[-1], t_gyr[-1] if t_gyr is not None else 1e18)
    if t1 - t0 < CFG["VENTANA"] / fs: return [], None
    tg = np.arange(t0, t1, 1 / fs)
    A = remuestrear(t_acc, acc, tg, fs)
    if unidades == "auto":
        unidades = "g" if np.nanmedian(np.linalg.norm(A, axis=1)) < 3 else "ms2"
    if unidades == "ms2": A = A / G0
    Gy = remuestrear(t_gyr, gyr, tg, fs) if t_gyr is not None else None
    Mg = remuestrear(t_mag, mag, tg, fs) if t_mag is not None else None
    val = ~np.isnan(A).any(1)
    if Gy is not None: val &= ~np.isnan(Gy).any(1)
    acc_c, gyr_c, mrel = calibrar_sesion(A, Gy, Mg)
    X, band = canales(acc_c, gyr_c, mrel, fs)
    return ventanas(X, val, CFG["VENTANA"], CFG["SOLAPE"]), band

def cargar_hmog(fs):
    cp = CACHE / f"hmog_{fs}hz_{CFG['MAX_VENT_HMOG']}.npz"
    if cp.exists(): z = np.load(cp); return z["W"], z["B"], z["S"]
    zips = sorted(Path(CFG["HMOG"]).glob("*.zip"))
    if SMOKE: zips = zips[:CFG["SMOKE_HMOG_USUARIOS"]]
    W, B, S = [], [], []
    por_usuario = CFG["MAX_VENT_HMOG"] // max(1, len(zips))
    for zi, zp in enumerate(zips):
        with zipfile.ZipFile(zp) as z:
            ses = sorted({n.split("/")[1] for n in z.namelist() if re.search(r"_session_\d+/Accelerometer.csv$", n)})
            if SMOKE: ses = ses[:CFG["SMOKE_HMOG_SESIONES"]]
            n_u = 0
            for s in ses:
                base = f"{zp.stem}/{s}/"
                try:
                    act = pd.read_csv(z.open(base + "Activity.csv"), header=None)
                    lee = lambda n: pd.read_csv(z.open(base + n), header=None, usecols=[1, 3, 4, 5]).to_numpy(np.float64)
                    a, g, m = lee("Accelerometer.csv"), lee("Gyroscope.csv"), lee("Magnetometer.csv")
                except Exception as e:
                    print("HMOG", base, "omitida:", type(e).__name__); continue
                for _, r in act.iterrows():
                    ini, fin = r[5] / 1e3, r[6] / 1e3          # relativos al EventTime, en s
                    sel = lambda x: x[(x[:, 0] / 1e9 >= ini) & (x[:, 0] / 1e9 <= fin)]
                    aa, gg, mm = sel(a), sel(g), sel(m)
                    if len(aa) < 300 or len(gg) < 300: continue
                    w, band = ventanas_de_tramo(aa[:, 0] / 1e9, aa[:, 1:], gg[:, 0] / 1e9, gg[:, 1:],
                                                mm[:, 0] / 1e9 if len(mm) > 10 else None, mm[:, 1:] if len(mm) > 10 else None, fs)
                    W += w; B += [band] * len(w); S += [f"hmog_{zp.stem}_{s}"] * len(w); n_u += len(w)
                if n_u >= por_usuario: break
        print(f"HMOG {zi + 1}/{len(zips)} {zp.stem}: {n_u} ventanas")
    W = np.stack(W).astype(np.float16); B = np.asarray(B, np.float32); S = np.asarray(S)
    np.savez(cp, W=W, B=B, S=S); return W, B, S

def cargar_extrasensory(fs):
    cp = CACHE / f"es_{fs}hz_{CFG['MAX_VENT_ES']}_{CFG['ES_MIN_POR_USUARIO']}.npz"
    if cp.exists(): z = np.load(cp); return z["W"], z["B"], z["S"]
    local = Path(CFG["ES_ACC"]).is_file() and Path(CFG["ES_GYR"]).is_file()
    if local: za, zg = zipfile.ZipFile(CFG["ES_ACC"]), zipfile.ZipFile(CFG["ES_GYR"])
    else: za, zg = zip_remoto(CFG["ES_ACC_URL"]), zip_remoto(CFG["ES_GYR_URL"])
    hilo = threading.local()          # un descriptor de archivo por hilo para leer en paralelo

    def miembro(c, info):
        if not local: return miembro_remoto(CFG[f"ES_{c}_URL"], info)
        if not hasattr(hilo, c): setattr(hilo, c, open(CFG[f"ES_{c}"], "rb"))
        return miembro_local(getattr(hilo, c), info)
    acc_n = {tuple(i.filename.split("/")[1:3]): i.filename for i in za.infolist() if i.filename.endswith(".dat")}
    gyr_n = {}
    for i in zg.infolist():
        if i.filename.endswith(".dat"):
            u, f = i.filename.split("/")[1:3]; gyr_n[(u, f.split(".")[0])] = i.filename
    pares = defaultdict(list)
    for (u, f), n in acc_n.items():
        k = (u, f.split(".")[0])
        if k in gyr_n: pares[u].append((n, gyr_n[k]))
    usuarios = sorted(pares)
    if SMOKE: usuarios = usuarios[:CFG["SMOKE_ES_USUARIOS"]]
    rng = np.random.default_rng(0)
    tareas = []
    for u in usuarios:
        lst = pares[u]; sel = rng.choice(len(lst), min(len(lst), CFG["ES_MIN_POR_USUARIO"]), replace=False)
        tareas += [(u, lst[j]) for j in sel]
    print(f"ExtraSensory: {len(usuarios)} usuarios, {len(tareas)} minutos a leer", "de los zips locales" if local else "del servidor")
    info_a = {i.filename: i for i in za.infolist()}; info_g = {i.filename: i for i in zg.infolist()}
    def leer(t):
        u, (na, ng) = t
        try:
            a = np.loadtxt(io.BytesIO(miembro("ACC", info_a[na])))
            g = np.loadtxt(io.BytesIO(miembro("GYR", info_g[ng])))
        except Exception:
            return u, [], None
        if a.ndim != 2 or g.ndim != 2 or len(a) < 200 or len(g) < 200: return u, [], None
        w, band = ventanas_de_tramo(a[:, 0], a[:, 1:4], g[:, 0], g[:, 1:4], None, None, fs, unidades="auto")
        return u, w, band
    W, B, S = [], [], []
    with ThreadPoolExecutor(16) as ex:
        for k, (u, w, band) in enumerate(ex.map(leer, tareas)):
            W += w; B += [band] * len(w); S += [f"es_{u}"] * len(w)
            if (k + 1) % 500 == 0: print(f"  {k + 1}/{len(tareas)} minutos, {len(W)} ventanas")
            if len(W) >= CFG["MAX_VENT_ES"]: break
    W = np.stack(W).astype(np.float16); B = np.asarray(B, np.float32); S = np.asarray(S)
    np.savez(cp, W=W, B=B, S=S); return W, B, S

def cargar_hhar(fs):
    """Ventanas de HHAR con el modelo de teléfono y el usuario (solo para la prueba de fuga)."""
    cp = CACHE / f"hhar_{fs}hz_{CFG['HHAR_VENT_POR_GRUPO']}_{CFG['HHAR_FILAS_POR_GRUPO']}.npz"
    if cp.exists(): z = np.load(cp); return z["W"], z["B"], z["MOD"], z["USR"]
    interno = CACHE / "hhar_actividad.zip"
    if not interno.exists():
        with zipfile.ZipFile(CFG["HHAR"]) as z, open(interno, "wb") as f: shutil.copyfileobj(z.open("Activity recognition exp.zip"), f)
    tablas = {}
    tope = CFG["HHAR_FILAS_POR_GRUPO"]
    with zipfile.ZipFile(interno) as z:
        for sensor in ["accelerometer", "gyroscope"]:
            n = f"Activity recognition exp/Phones_{sensor}.csv"; partes = []; cuenta = Counter()
            for ch in pd.read_csv(z.open(n), usecols=["Creation_Time", "x", "y", "z", "User", "Model", "Device"],
                                  dtype={"x": np.float32, "y": np.float32, "z": np.float32, "User": str, "Model": str, "Device": str},
                                  chunksize=1_000_000):
                ch["k"] = ch.Device + "|" + ch.User
                ch["r"] = ch.groupby("k").cumcount() + ch.k.map(cuenta).fillna(0).astype(int)
                for k, v in ch.groupby("k").size().items(): cuenta[k] += v
                partes.append(ch[ch.r < tope].drop(columns=["k", "r"]))
            tablas[sensor] = pd.concat(partes, ignore_index=True)
    W, B, MOD, USR = [], [], [], []
    ga = tablas["accelerometer"].groupby(["Device", "User"], observed=True)
    gg = dict(list(tablas["gyroscope"].groupby(["Device", "User"], observed=True)))
    for clave, da in ga:
        if clave not in gg: continue
        dg = gg[clave]
        da = da.sort_values("Creation_Time"); dg = dg.sort_values("Creation_Time")
        ta, tg_ = da.Creation_Time.to_numpy() / 1e9, dg.Creation_Time.to_numpy() / 1e9
        cortes = np.where(np.diff(ta) > 1.0)[0] + 1; n_g = 0
        for seg in np.split(np.arange(len(ta)), cortes):
            if len(seg) < 500: continue
            t0, t1 = ta[seg[0]], ta[seg[-1]]; sg = (tg_ >= t0) & (tg_ <= t1)
            if sg.sum() < 500: continue
            w, band = ventanas_de_tramo(ta[seg], da[["x", "y", "z"]].to_numpy()[seg], tg_[sg], dg[["x", "y", "z"]].to_numpy()[sg], None, None, fs)
            w = w[:CFG["HHAR_VENT_POR_GRUPO"] - n_g]
            W += w; B += [band] * len(w); MOD += [str(da.Model.iloc[0])] * len(w); USR += [str(clave[1])] * len(w); n_g += len(w)
            if n_g >= CFG["HHAR_VENT_POR_GRUPO"]: break
    W = np.stack(W).astype(np.float16); B = np.asarray(B, np.float32)
    np.savez(cp, W=W, B=B, MOD=np.asarray(MOD), USR=np.asarray(USR)); return W, B, np.asarray(MOD), np.asarray(USR)

FS = CFG["FS"]
PUBLICOS = {}
if CFG["USAR_HMOG"] and Path(CFG["HMOG"]).exists(): PUBLICOS["hmog"] = cargar_hmog(FS)
if CFG["USAR_ES"]:
    try: PUBLICOS["extrasensory"] = cargar_extrasensory(FS)
    except Exception as e: print("ExtraSensory no disponible:", e)
HHAR = cargar_hhar(FS) if CFG["USAR_HHAR"] and Path(CFG["HHAR"]).exists() else None
for k, (W, B, S) in PUBLICOS.items():
    print(f"{k}: {len(W)} ventanas de {len(set(S))} sesiones; con giroscopio {B[:, 0].mean():.2f}, con magnetómetro {B[:, 1].mean():.2f}")
if HHAR is not None: print("HHAR:", len(HHAR[0]), "ventanas;", Counter(HHAR[2]))

## 4. Motion ID (Gavron et al., 2023)

Se lee de la copia local (`MID_LOCAL`, viene en `pickupauth_datos.zip`) o, si no está, se baja de Kaggle sin
cuenta. Los sensores vienen en binario (registros de 20 bytes: marca de tiempo int64
big-endian en ms y tres float32 little-endian), a 50 Hz, con `screen.txt` (`SCREEN_ON`, `SCREEN_OFF`,
`USER_PRESENT`) en el mismo reloj.

- **Movimiento específico** (condiciones ideales): 101 personas, un mismo Samsung S20, 300 tomas desde la mesa
  cada una, desbloqueo con huella. Como las tomas están a ~5 s una de otra, la ventana va de −3 s (como en el
  artículo) hasta el siguiente `SCREEN_OFF`. La pantalla suele encenderse al empezar a levantar el teléfono
  (mediana: 3,4 s antes del desbloqueo; en ~1 de cada 4 tomas, a la vez que el desbloqueo con la huella), así que
  casi todo el gesto cae en el tramo de asentamiento y la toma suele quedar vacía.
- **Todos los movimientos** (campo): 6 personas × 6 Galaxy S10e, rotando cada 2 semanas. Se recorta −8 s a +5 s
  alrededor de cada `USER_PRESENT`, igual que el dataset propio. Lo bajado de Kaggle se borra después; la copia
  local no se toca.

In [ ]:
MID_DT = np.dtype([("t", ">i8"), ("x", "<f4"), ("y", "<f4"), ("z", "<f4")])
MID_ESP = "motionid-imu-specific-motion"
MID_CAMPO = ["motionid-imu-all-motions-part1", "motionid-imu-all-motions-part2", "motionid-imu-all-motions-part3"]

def kaggle_lista(slug):
    copia = Path(CFG["MID_LOCAL"]) / f"{slug}.json"          # listado guardado junto a la copia local
    if copia.exists(): return [tuple(x) for x in json.loads(copia.read_text())]
    out, tok = [], None
    while True:
        u = CFG["MID_LISTA"].format(slug=slug) + (f"&pageToken={urllib.parse.quote(tok)}" if tok else "")
        d = json.load(urllib.request.urlopen(u, timeout=60))
        out += [(f["name"], f["totalBytes"]) for f in d.get("datasetFiles", [])]
        tok = d.get("nextPageTokenNullable")
        if not tok or not d.get("datasetFiles"): return out

def kaggle_bajar(slug, ruta, destino):
    destino = Path(destino)
    if destino.exists(): return destino
    destino.parent.mkdir(parents=True, exist_ok=True)
    url = CFG["MID_URL"].format(slug=slug, path=urllib.parse.quote(ruta, safe=""))
    tmp = destino.with_suffix(".parte")
    for intento in range(5):
        try:
            with urllib.request.urlopen(url, timeout=600) as r, open(tmp, "wb") as f: shutil.copyfileobj(r, f, 1 << 20)
            break
        except Exception:
            if intento == 4: raise
            time.sleep(5 * 2 ** intento)
    with open(tmp, "rb") as f: es_zip = f.read(4) == b"PK\x03\x04"
    if es_zip:
        with zipfile.ZipFile(tmp) as z, open(destino, "wb") as f: shutil.copyfileobj(z.open(z.namelist()[0]), f, 1 << 20)
        tmp.unlink()
    else:
        tmp.rename(destino)
    return destino

def mid_local(slug, c):
    """Carpeta de la copia local de Motion ID, si tiene lo necesario; si no, None."""
    loc = Path(CFG["MID_LOCAL"]) / slug / c
    return loc if (loc / "accel.txt").exists() and (loc / "screen.txt").exists() else None

def mid_sensor(p):
    a = np.memmap(p, dtype=MID_DT, mode="r") if Path(p).exists() and Path(p).stat().st_size >= 20 else np.zeros(0, MID_DT)
    return a

def mid_pantalla(p, ancla_ms=None):
    filas = []
    delta_h = None
    for linea in Path(p).read_text(errors="ignore").splitlines():
        if linea.startswith("STOP") or not linea.strip(): continue
        partes = linea.split()
        if len(partes) == 2:
            filas.append((int(partes[0]), partes[1].split(".")[-1]))
        elif len(partes) == 3 and ancla_ms is not None:      # formato viejo: fecha hora evento (sin zona)
            ts = pd.Timestamp(partes[0] + " " + partes[1])
            if delta_h is None:
                cand = np.arange(-23, 24)
                delta_h = int(cand[np.argmin([abs((ts + pd.Timedelta(hours=int(h))).value // 10**6 - ancla_ms) for h in cand])])
            filas.append(((ts + pd.Timedelta(hours=delta_h)).value // 10**6, partes[2].split(".")[-1]))
    return pd.DataFrame(filas, columns=["t_ms", "evento"]).sort_values("t_ms").reset_index(drop=True)

def mid_episodios(carpeta, modo, fs):
    """Episodios de una carpeta con accel/gyro/MagneticField/screen. modo: 'especifico' o 'campo'."""
    c = Path(carpeta)
    A, G, M = mid_sensor(c / "accel.txt"), mid_sensor(c / "gyro.txt"), mid_sensor(c / "MagneticField.txt")
    if len(A) < 100: return []
    pant = mid_pantalla(c / "screen.txt", ancla_ms=int(A["t"][0]))
    ups = pant.t_ms[pant.evento == "USER_PRESENT"].to_numpy()
    offs = pant.t_ms[pant.evento == "SCREEN_OFF"].to_numpy(); ons = pant.t_ms[pant.evento == "SCREEN_ON"].to_numpy()
    pre, post = (3.0, 2.0) if modo == "especifico" else (CFG["PRE_S"], CFG["POST_S"])
    # Las marcas vienen en orden de bytes big-endian dentro de un registro: se convierten una sola vez a int64 nativo.
    # Sin esto, cada searchsorted copia el arreglo completo (decenas de millones de muestras en los datos de campo).
    ta = np.asarray(A["t"], dtype=np.int64); tg_ = np.asarray(G["t"], dtype=np.int64); tm = np.asarray(M["t"], dtype=np.int64)
    out = []
    for k, up in enumerate(ups):
        if k > 0 and up - ups[k - 1] < 2000: continue
        desde = up - (CFG["PRE_S"] + 1) * 1000; hasta = up + (CFG["POST_S"] + 1) * 1000
        sl = lambda t: slice(np.searchsorted(t, desde), np.searchsorted(t, hasta))
        sa, sg, sm = sl(ta), sl(tg_), sl(tm)
        if sa.stop - sa.start < 50: continue
        a = np.stack([A["x"][sa], A["y"][sa], A["z"][sa]], 1).astype(np.float64)
        g = np.stack([G["x"][sg], G["y"][sg], G["z"][sg]], 1).astype(np.float64) if sg.stop > sg.start + 50 else None
        m = np.stack([M["x"][sm], M["y"][sm], M["z"][sm]], 1).astype(np.float64) if sm.stop > sm.start + 50 else None
        off_ant = offs[offs < up]; off_sig = offs[offs > up]; on_cerca = ons[(ons > up - 5000) & (ons < up + 1000)]
        v_desde = max(up - pre * 1000, off_ant[-1] if len(off_ant) else -1e18) / 1e3
        v_hasta = min(up + post * 1000, off_sig[0] if len(off_sig) else 1e18) / 1e3
        t_on = on_cerca[0] / 1e3 if len(on_cerca) else None
        r, mot = armar_episodio(ta[sa] / 1e3, a, tg_[sg] / 1e3 if g is not None else None, g,
                                tm[sm] / 1e3 if m is not None else None, m, t_up=up / 1e3, t_on=t_on, t_toma=None,
                                fs=fs, pre_s=CFG["PRE_S"], post_s=CFG["POST_S"], valido_desde=v_desde, valido_hasta=v_hasta)
        if r is None: continue
        r["t_up_ms"] = int(up); r["intento"] = k
        out.append(r)
    return out

def mid_a_arreglos(eps, extra):
    meta = pd.DataFrame([{k: v for k, v in e.items() if k not in ("X", "val", "fases", "banderas")} | extra(e) for e in eps])
    return (meta, np.stack([e["X"] for e in eps]), np.stack([e["val"] for e in eps]),
            np.stack([e["fases"] for e in eps]), np.asarray([e["banderas"] for e in eps], np.float32))

def cargar_mid_especifico(fs):
    cp = CACHE / f"mid_esp_{fs}hz_{'smoke' if SMOKE else 'full'}.npz"
    if cp.exists():
        z = np.load(cp); return pd.read_parquet(str(cp).replace(".npz", ".parquet")), z["X"], z["val"], z["fases"], z["banderas"]
    archivos = kaggle_lista(MID_ESP)
    carpetas = sorted({"/".join(n.split("/")[:-1]) for n, _ in archivos})
    if SMOKE:
        ent = [c for c in carpetas if "/train_val_test/" in c][:CFG["SMOKE_MID_ENTRENAR"]]
        pru = [c for c in carpetas if "/additional_test/" in c][:CFG["SMOKE_MID_PRUEBA"]]
        carpetas = ent + pru
    eps = []
    for i, c in enumerate(carpetas):
        loc = mid_local(MID_ESP, c)
        if loc is None:
            loc = CACHE / "mid" / c
            for f in ["accel.txt", "gyro.txt", "MagneticField.txt", "screen.txt"]:
                kaggle_bajar(MID_ESP, f"{c}/{f}", loc / f)
        grupo, usuario = c.split("/")[1], c.split("/")[2]
        e = mid_episodios(loc, "especifico", fs)
        for x in e: x.update(sujeto=f"mid{usuario}", grupo=grupo, origen="SURFACE", tipo="unlock", impostor=False,
                             telefono="S20", fecha=pd.Timestamp(x["t_up_ms"], unit="ms"))
        eps += e
        print(f"Motion ID específico {i + 1}/{len(carpetas)} {grupo}/{usuario}: {len(e)} tomas")
    res = mid_a_arreglos(eps, lambda e: {})
    np.savez(cp, X=res[1], val=res[2], fases=res[3], banderas=res[4]); res[0].to_parquet(str(cp).replace(".npz", ".parquet"))
    return res

MID_E = cargar_mid_especifico(FS) if CFG["USAR_MID_ESPECIFICO"] else None
if MID_E is not None:
    print(MID_E[0].groupby("grupo").sujeto.nunique(), "| tomas por persona (mediana):", MID_E[0].groupby("sujeto").size().median())

In [ ]:
def cargar_mid_campo(fs):
    """6 personas × 6 S10e. Se procesa un par (persona, teléfono) a la vez y se borra lo crudo."""
    cp = CACHE / f"mid_campo_{fs}hz_{'smoke' if SMOKE else 'full'}.npz"
    if cp.exists():
        z = np.load(cp); return pd.read_parquet(str(cp).replace(".npz", ".parquet")), z["X"], z["val"], z["fases"], z["banderas"]
    eps = []
    if SMOKE:
        # En SMOKE no se bajan ~150 GB: se reutilizan carpetas del movimiento específico (mismo formato) como si
        # fueran pares persona-teléfono, solo para recorrer el código.
        base = sorted([*(CACHE / "mid").rglob("accel.txt"), *(Path(CFG["MID_LOCAL"]) / MID_ESP).rglob("accel.txt")])[:4]
        pares = [(p.parent, (i % 2) + 1, i // 2) for i, p in enumerate(base)]
        for carpeta, usuario, tel in pares:
            e = mid_episodios(carpeta, "campo", fs)
            for x in e: x.update(sujeto=f"midc{usuario}", telefono=f"s10e_{tel}", origen="UNKNOWN", tipo="unlock",
                                 impostor=False, fecha=pd.Timestamp(x["t_up_ms"], unit="ms"))
            eps += e
    else:
        for slug in MID_CAMPO:
            archivos = kaggle_lista(slug)
            carpetas = sorted({"/".join(n.split("/")[:-1]) for n, _ in archivos})
            for c in carpetas:
                partes = c.split("/"); usuario = int(partes[1]); tel = int(partes[2].split("#")[-1])
                if CFG["MID_CAMPO_PARES"] and (usuario, tel) not in CFG["MID_CAMPO_PARES"]: continue
                marca = RES / "datos" / f"midcampo_{usuario}_{tel}.npz"
                if marca.exists():
                    z = np.load(marca, allow_pickle=True); eps += list(z["eps"]); continue
                loc = mid_local(slug, c)
                if loc is not None:
                    print(f"Motion ID campo: persona {usuario}, teléfono {tel}: copia local")
                else:
                    loc = CACHE / "mid_campo_tmp"
                    shutil.rmtree(loc, ignore_errors=True)
                    tam = sum(b for n, b in archivos if n.startswith(c + "/") and n.split("/")[-1] in ("accel.txt", "gyro.txt", "MagneticField.txt"))
                    print(f"Motion ID campo: persona {usuario}, teléfono {tel}: bajando ~{tam / 1e9:.1f} GB…")
                    for f in ["accel.txt", "gyro.txt", "MagneticField.txt", "screen.txt"]:
                        if any(n == f"{c}/{f}" for n, _ in archivos): kaggle_bajar(slug, f"{c}/{f}", loc / f)
                e = mid_episodios(loc, "campo", fs)
                for x in e: x.update(sujeto=f"midc{usuario}", telefono=f"s10e_{tel}", origen="UNKNOWN", tipo="unlock",
                                     impostor=False, fecha=pd.Timestamp(x["t_up_ms"], unit="ms"))
                np.savez(marca, eps=np.array(e, dtype=object)); eps += e
                if loc == CACHE / "mid_campo_tmp": shutil.rmtree(loc, ignore_errors=True)   # nunca la copia local
                print(f"  {len(e)} desbloqueos")
    res = mid_a_arreglos(eps, lambda e: {})
    np.savez(cp, X=res[1], val=res[2], fases=res[3], banderas=res[4]); res[0].to_parquet(str(cp).replace(".npz", ".parquet"))
    return res

MID_C = cargar_mid_campo(FS) if CFG["USAR_MID_CAMPO"] else None
if MID_C is not None: print(MID_C[0].groupby(["sujeto", "telefono"]).size().unstack(fill_value=0))

## 5. Particiones y conjuntos de episodios

- Por persona: 60 % entrenamiento, 20 % validación, 20 % prueba. Se guardan en `particiones.json` y no se tocan.
  Si aparece un participante nuevo, se le asigna un grupo con un hash de su código (determinista) y se agrega.
- **Las rondas R1–R4 deciden con las personas de validación.** Las de prueba solo se miden en la ronda 5, en las
  líneas base y en la comparación final, para que la elección del modelo no vea la prueba.
- Dentro de cada persona de validación o prueba, en orden temporal: enrolamiento, calibración del umbral y
  evaluación.

In [ ]:
def particiones(meta):
    p = RES / "particiones.json"
    part = json.loads(p.read_text()) if p.exists() else {}
    cuenta = meta[meta.tipo.isin(["unlock", "in_place"]) & ~meta.impostor].groupby("sujeto").size()
    elegibles = sorted(cuenta[cuenta >= CFG["MIN_EPISODIOS"]].index)
    nuevos = [s for s in elegibles if s not in part]
    if not part and nuevos:
        rng = np.random.default_rng(12345); orden = list(rng.permutation(nuevos))
        n = len(orden); a = max(1, round(n * CFG["PROP_PARTICION"][0])); b = max(1, round(n * CFG["PROP_PARTICION"][1]))
        for i, s in enumerate(orden): part[s] = "entrenamiento" if i < a else ("validacion" if i < a + b else "prueba")
    else:
        for s in nuevos:
            h = int(hashlib.md5(s.encode()).hexdigest(), 16) % 100 / 100
            part[s] = "entrenamiento" if h < CFG["PROP_PARTICION"][0] else ("validacion" if h < sum(CFG["PROP_PARTICION"][:2]) else "prueba")
    guardar_json(part, p)
    return part

PART = particiones(META)
print(Counter(PART.values()), {g: sorted(s for s, v in PART.items() if v == g) for g in ["validacion", "prueba"]})

class Conjunto:
    """Episodios en memoria (unidades físicas) + metadatos. Se indexa con arreglos de posiciones."""
    def __init__(s, meta, X, val, fases, banderas, nombre):
        s.meta = meta.reset_index(drop=True); s.X = torch.from_numpy(np.ascontiguousarray(X)); s.val = torch.from_numpy(val)
        s.fases = torch.from_numpy(fases.astype(np.int8)); s.band = torch.from_numpy(banderas); s.nombre = nombre
        s.origen = torch.tensor([ORIGEN_ID.get(o, ORIGEN_ID["UNKNOWN"]) for o in s.meta.origen], dtype=torch.long)
        s.i_toma = torch.tensor(s.meta.i_toma.to_numpy(), dtype=torch.long)
    def __len__(s): return len(s.meta)

if SMOKE:
    # Solo para probar el código del mismo teléfono: todavía no hay sesiones de laboratorio, así que se copian episodios
    # de una persona de entrenamiento al teléfono de cada persona evaluada, marcados como impostor. NO se usa en FULL.
    donante = sorted(s for s, g in PART.items() if g == "entrenamiento")[0]
    extra_m, extra_i = [], []
    for s, g in PART.items():
        if g == "entrenamiento": continue
        i = np.where((META.sujeto == donante) & (META.tipo == "unlock"))[0][:12]
        fs_dueno = META.fecha[META.sujeto == s].sort_values()
        d = META.iloc[i].copy(); d["sujeto"] = s; d["impostor"] = True
        d["fecha"] = fs_dueno.iloc[len(fs_dueno) * 2 // 3] + pd.to_timedelta(np.arange(len(d)), unit="min")
        extra_m.append(d); extra_i.append(i)
    if extra_m:
        ii = np.concatenate(extra_i)
        META = pd.concat([META] + extra_m, ignore_index=True)
        XP, VP, FP, BP = (np.concatenate([a, a[ii]]) for a in (XP, VP, FP, BP))
        print("SMOKE: impostores simulados en el mismo teléfono:", len(ii))

PROPIO = Conjunto(META, XP, VP, FP, BP, "propio")
del XP, VP, FP, BP

def stats_persona(cj, idx):
    """Media y desvío por canal sobre las muestras válidas de los episodios idx (sin banderas)."""
    X = cj.X[idx].float(); v = cj.val[idx].float()[:, None, :]
    n = v.sum((0, 2)).clamp(min=1)
    mu = (X * v).sum((0, 2)) / n
    sd = (((X - mu[None, :, None]) ** 2) * v).sum((0, 2)).div(n).sqrt().clamp(min=1e-3)
    if CFG["NORM"] == "escala": mu = torch.zeros_like(mu)
    return mu, sd

def indices(cj, sujeto=None, tipos=("unlock", "in_place"), impostor=False):
    m = cj.meta
    sel = m.tipo.isin(tipos)
    if sujeto is not None: sel &= m.sujeto == sujeto
    if impostor is not None: sel &= m.impostor == impostor
    return np.where(sel)[0]

def orden_temporal(cj, idx): return idx[np.argsort(cj.meta.fecha.to_numpy()[idx], kind="stable")]

def impostores_otros(cj, s, otros):
    """Últimos episodios de las demás personas del grupo (impostores de cero esfuerzo con su propio teléfono)."""
    l = [orden_temporal(cj, indices(cj, o))[-CFG["MAX_IMPOSTOR_OTROS"]:] for o in otros if o != s]
    return np.concatenate(l) if l else np.zeros(0, dtype=np.int64)

## 6. Aumentos y preparación del lote

Lo que el modelo debe aprender a ignorar: sesgo y ganancia por eje, ruido, estiramiento leve del tiempo (±10 %),
rotaciones pequeñas (≤ 15°) y borrado de un sensor. No se usan rotaciones grandes, porque borrarían el ángulo
en que cada persona sujeta el teléfono. Los aumentos se aplican en unidades físicas y después se normaliza con
las estadísticas de la persona dueña del modelo.

In [ ]:
def rotaciones(B, max_grados, dev):
    eje = F.normalize(torch.randn(B, 3, device=dev), dim=1)
    ang = (torch.rand(B, device=dev) * 2 - 1) * math.radians(max_grados)
    K = torch.zeros(B, 3, 3, device=dev)
    K[:, 0, 1], K[:, 0, 2], K[:, 1, 2] = -eje[:, 2], eje[:, 1], -eje[:, 0]
    K = K - K.transpose(1, 2)
    I = torch.eye(3, device=dev).expand(B, 3, 3)
    s, c = torch.sin(ang)[:, None, None], torch.cos(ang)[:, None, None]
    return I + s * K + (1 - c) * (K @ K)

def estirar(x, fases, val, centro, factor):
    """Estira el tiempo alrededor de `centro` (índice) por `factor` (B,). Interpolación lineal; fases por vecino."""
    B, C, T = x.shape; dev = x.device
    t = torch.arange(T, device=dev, dtype=torch.float32)[None]
    fuente = centro[:, None].float() + (t - centro[:, None].float()) * factor[:, None]
    dentro = (fuente >= 0) & (fuente <= T - 1)
    f0 = fuente.clamp(0, T - 1).floor().long(); f1 = (f0 + 1).clamp(max=T - 1); w = (fuente.clamp(0, T - 1) - f0.float())[:, None]
    g = lambda z, i: torch.gather(z, 2, i[:, None].expand(-1, z.shape[1], -1))
    x2 = g(x, f0) * (1 - w) + g(x, f1) * w
    near = fuente.clamp(0, T - 1).round().long()
    fa2 = torch.gather(fases.long(), 1, near).to(fases.dtype); v2 = torch.gather(val, 1, near) & dentro
    fa2[~dentro] = -1
    return x2, fa2, v2

def aumentar(x, fases, val, band, centro, estirar_t=True, max_grados=15):
    B = x.shape[0]; dev = x.device
    x = x.clone(); band = band.clone()
    R = rotaciones(B, max_grados, dev)
    for a in (0, 3, 6): x[:, a:a + 3] = torch.einsum("bij,bjt->bit", R, x[:, a:a + 3])
    x[:, 0:3] = x[:, 0:3] * (1 + 0.03 * (2 * torch.rand(B, 3, 1, device=dev) - 1)) + 0.02 * torch.randn(B, 3, 1, device=dev)
    x[:, 3:6] = x[:, 3:6] * (1 + 0.03 * (2 * torch.rand(B, 3, 1, device=dev) - 1)) + 0.01 * torch.randn(B, 3, 1, device=dev)
    x[:, :9] = x[:, :9] + 0.01 * torch.randn_like(x[:, :9]) * x[:, :9].std(2, keepdim=True)
    if estirar_t:
        x, fases, val = estirar(x, fases, val, centro, 1 + 0.1 * (2 * torch.rand(B, device=dev) - 1))
    borra = torch.rand(B, device=dev)
    sin_g = borra < 0.1; sin_m = (borra >= 0.1) & (borra < 0.2)
    x[sin_g, 3:6] = 0; band[sin_g, 0] = 0
    x[sin_m, 9] = 0; band[sin_m, 1] = 0
    return x, fases, val, band

def preparar(x, val, band, mu, sd):
    """Normaliza (con las estadísticas de la persona dueña), pone en cero lo inválido y agrega las banderas."""
    xn = (x - mu[..., None]) / sd[..., None]
    xn = xn * val[:, None].float()
    xn[:, 3:6] = xn[:, 3:6] * band[:, 0, None, None]
    xn[:, 9] = xn[:, 9] * band[:, 1, None]
    T = x.shape[2]
    flags = band[:, :, None].expand(-1, 2, T) * val[:, None].float()
    return torch.cat([xn, flags], 1)

def truncar_x(fases, val, i_up, x_s, fs):
    """Corta el uso en x segundos (x_s escalar o tensor (B,)): fases −1 y no válido después de i_up + x."""
    T = fases.shape[1]; t = torch.arange(T, device=fases.device)[None]
    lim = i_up + torch.as_tensor(x_s, device=fases.device).float().reshape(-1, 1) * fs
    fuera = (t >= lim).expand_as(fases)
    fases = fases.clone(); fases[fuera] = -1
    return fases, val & ~fuera

## 7. Arquitecturas

Capas bajas (lo que se preentrena en la etapa 1), una por familia de la ronda 1. Todas reciben (lote, 12, T) y
devuelven (lote, D, T / paso). Las basadas en transformer agrupan de a 5 muestras (paso 5); las demás conservan
la resolución. Todas empiezan con normalización por lotes sobre la entrada, que es lo que ajusta el nivel
"normalización" de la ronda 3.

Encima, el modelo de episodio: una rama por tramo (toma; asentamiento y desbloqueo; uso) con su propio bloque
y agrupamiento con atención sobre su tramo, y una fusión con atención que recibe el origen de la toma como un
token de contexto. Los adaptadores (cuello de botella residual, inicializado en cero) existen siempre pero solo
se entrenan en el nivel "adaptadores".

In [ ]:
class BloqueRes(nn.Module):
    def __init__(s, d, k=5, dil=1, causal=False):
        super().__init__(); s.pad, s.causal = (k - 1) * dil, causal
        s.c1, s.b1 = nn.Conv1d(d, d, k, dilation=dil), nn.BatchNorm1d(d)
        s.c2, s.b2 = nn.Conv1d(d, d, k, dilation=dil), nn.BatchNorm1d(d)
    def _p(s, x): return F.pad(x, (s.pad, 0)) if s.causal else F.pad(x, (s.pad // 2, s.pad - s.pad // 2))
    def forward(s, x):
        h = F.gelu(s.b1(s.c1(s._p(x)))); return F.gelu(x + s.b2(s.c2(s._p(h))))

class CodPos(nn.Module):
    def __init__(s, d, n=4096):
        super().__init__(); p = torch.arange(n)[:, None]; i = torch.arange(0, d, 2)
        pe = torch.zeros(n, d); pe[:, 0::2] = torch.sin(p / 10000 ** (i / d)); pe[:, 1::2] = torch.cos(p / 10000 ** (i / d))
        s.register_buffer("pe", pe)
    def forward(s, x): return x + s.pe[: x.shape[1]][None]

def capa_tf(d, n): return nn.TransformerEncoder(nn.TransformerEncoderLayer(d, 4, 2 * d, 0.1, batch_first=True, norm_first=True), n)

class Bajas(nn.Module):
    def __init__(s, tipo, c=C_ENT, d=64):
        super().__init__(); s.tipo, s.paso, s.d = tipo, 1, d
        s.bn_in = nn.BatchNorm1d(c)
        tallo = lambda: nn.Sequential(nn.Conv1d(c, d, 7, padding=3), nn.BatchNorm1d(d), nn.GELU())
        if tipo == "cnn":
            s.red = nn.Sequential(tallo(), BloqueRes(d, 5), BloqueRes(d, 5), BloqueRes(d, 5, dil=2))
        elif tipo == "tcn":
            s.red = nn.Sequential(nn.Conv1d(c, d, 1), *[BloqueRes(d, 3, dil=2 ** i, causal=True) for i in range(5)])
        elif tipo == "gru":
            s.proy = nn.Linear(c, d); s.rnn = nn.GRU(d, d, 2, batch_first=True); s.bn_out = nn.BatchNorm1d(d)
        elif tipo == "cnn_gru":
            s.red = nn.Sequential(tallo(), BloqueRes(d, 5), BloqueRes(d, 5)); s.rnn = nn.GRU(d, d, 1, batch_first=True); s.bn_out = nn.BatchNorm1d(d)
        elif tipo == "transformer":
            s.paso = 5; s.parche = nn.Sequential(nn.Conv1d(c, d, 5, stride=5), nn.BatchNorm1d(d)); s.pos = CodPos(d); s.tf = capa_tf(d, 2)
        elif tipo == "cnn_transformer":
            s.paso = 5; s.red = nn.Sequential(tallo(), BloqueRes(d, 5), BloqueRes(d, 5)); s.pos = CodPos(d); s.tf = capa_tf(d, 1)
        else: raise ValueError(tipo)
    def forward(s, x):
        x = s.bn_in(x)
        if s.tipo in ("cnn", "tcn"): return s.red(x)
        if s.tipo == "gru":
            h, _ = s.rnn(s.proy(x.transpose(1, 2))); return s.bn_out(h.transpose(1, 2))
        if s.tipo == "cnn_gru":
            h, _ = s.rnn(s.red(x).transpose(1, 2)); return s.bn_out(h.transpose(1, 2))
        if s.tipo == "transformer":
            T = x.shape[2] // 5 * 5
            h = s.parche(x[:, :, :T]).transpose(1, 2); return s.tf(s.pos(h)).transpose(1, 2)
        if s.tipo == "cnn_transformer":
            T = x.shape[2] // 5 * 5
            h = F.avg_pool1d(s.red(x[:, :, :T]), 5).transpose(1, 2); return s.tf(s.pos(h)).transpose(1, 2)

def bajar_etiquetas(f, paso, Tp):
    """Lleva etiquetas por muestra (B,T) a la resolución de las capas bajas (B,Tp) tomando la muestra central."""
    if paso == 1: return f[:, :Tp]
    idx = torch.arange(Tp, device=f.device) * paso + paso // 2
    return f[:, idx.clamp(max=f.shape[1] - 1)]

class Adaptador(nn.Module):
    def __init__(s, d, r=8):
        super().__init__(); s.baja, s.sube = nn.Linear(d, r), nn.Linear(r, d)
        nn.init.zeros_(s.sube.weight); nn.init.zeros_(s.sube.bias)
    def forward(s, h): return h + s.sube(F.gelu(s.baja(h.transpose(1, 2)))).transpose(1, 2)

class Rama(nn.Module):
    def __init__(s, d):
        super().__init__(); s.bloque, s.adap, s.q = BloqueRes(d, 5), Adaptador(d), nn.Linear(d, 1)
        s.vacio = nn.Parameter(torch.zeros(d))
    def forward(s, h, m):
        g = s.adap(s.bloque(h))
        a = s.q(g.transpose(1, 2)).squeeze(-1).masked_fill(~m, -1e4)
        w = torch.softmax(a, 1) * m.float()
        v = (g * w[:, None]).sum(-1) / w.sum(1, keepdim=True).clamp(min=1e-6)
        return torch.where(m.any(1, keepdim=True), v, s.vacio.expand_as(v))

def agrupar(h, m):
    """Promedio de h (B,D,T) sobre la máscara m (B,T)."""
    w = m.float(); return (h * w[:, None]).sum(-1) / w.sum(1, keepdim=True).clamp(min=1)

class ModeloEpisodio(nn.Module):
    def __init__(s, bajas, d, dz):
        super().__init__()
        s.bajas, s.adap_bajas = bajas, Adaptador(d)
        s.ramas = nn.ModuleList([Rama(d) for _ in range(3)])
        s.emb_fase = nn.Parameter(torch.randn(3, d) * 0.02); s.emb_origen = nn.Embedding(len(ORIGENES), d)
        s.fusion = capa_tf(d, 1); s.salida = nn.Linear(d, dz)
        # tareas auxiliares de la etapa 2 (sin identidad)
        s.cab_fases = nn.Conv1d(d, 4, 1); s.cab_real = nn.Linear(d, 2); s.cab_origen = nn.Linear(d, 3)
    def forward(s, x, fases, origen, ramas_on=None, i_toma=None):
        h = s.adap_bajas(s.bajas(x)); Tp = h.shape[2]; f = bajar_etiquetas(fases, s.bajas.paso, Tp)
        vec, val = [], []
        for p, r in enumerate(s.ramas):
            m = f == p + 1; vec.append(r(h, m)); val.append(m.any(1))
        V = torch.stack(vec, 1) + s.emb_fase; val = torch.stack(val, 1)
        if ramas_on is not None: val = val & ramas_on
        tok = torch.cat([V, s.emb_origen(origen)[:, None]], 1)
        pad = torch.cat([~val, torch.zeros_like(val[:, :1])], 1)
        o = s.fusion(tok, src_key_padding_mask=pad)
        w = val.float()
        z = s.salida((o[:, :3] * w[..., None]).sum(1) / w.sum(1, keepdim=True).clamp(min=1))
        out = dict(z=z, h=h, V=V, validas=val, f=f)
        if i_toma is not None:      # región de la toma para la tarea real/falso: [inicio, inicio + 5 s]
            t = torch.arange(Tp, device=x.device)[None] * s.bajas.paso
            ini = i_toma[:, None]; reg = (t >= ini) & (t < ini + 5 * FS)
            out["real"] = s.cab_real(agrupar(h, reg))
        return out

def proyector(d_in, d, capas=3):
    mods, a = [], d_in
    for i in range(capas - 1): mods += [nn.Linear(a, d), nn.BatchNorm1d(d), nn.ReLU()]; a = d
    return nn.Sequential(*mods, nn.Linear(a, d))

class Cabezas(nn.Module):
    """Cabezas del dueño y del preentrenamiento sobre z."""
    def __init__(s, dz, d, T_red):
        super().__init__()
        s.proy = proyector(dz, CFG["PROY"])
        s.dec = nn.Sequential(nn.Linear(dz, 256), nn.GELU(), nn.Linear(256, C_DATOS * T_red))   # autocodificador
        s.pred = nn.Sequential(nn.Linear(2 * d, 2 * d), nn.GELU(), nn.Linear(2 * d, d))          # uso desde toma + asentamiento
        s.svdd = nn.Sequential(nn.Linear(dz, 128, bias=False), nn.GELU(), nn.Linear(128, 64, bias=False))
        s.lineal = nn.Linear(dz, dz); nn.init.eye_(s.lineal.weight); nn.init.zeros_(s.lineal.bias)  # "cabeza" del conjunto
        s.T_red = T_red

def n_param(m): return sum(p.numel() for p in m.parameters())

for a in CFG["ARQUITECTURAS"]:
    b = Bajas(a, d=CFG["D"]); m = ModeloEpisodio(b, CFG["D"], CFG["DZ"])
    x = torch.randn(2, C_ENT, 650); f = torch.randint(-1, 4, (2, 650)).to(torch.int8)
    o = m(x, f, torch.zeros(2, dtype=torch.long), i_toma=torch.tensor([100, 200]))
    print(f"{a:16s} bajas {n_param(b):>8,d} parámetros | episodio {n_param(m):>8,d} | z {tuple(o['z'].shape)} | paso {b.paso}")

## 8. Objetivos autosupervisados

Barlow Twins (punto de partida, λ = 0,005), VICReg, reconstrucción enmascarada (se tapa el 30–40 % de la señal
en tramos) y predicción entre sensores (el giroscopio a partir del acelerómetro, la gravedad y el magnetómetro).
Ninguno usa negativos, así que ninguno premia distinguir teléfonos.

In [ ]:
def perdida_barlow(z1, z2, lam=None):
    lam = CFG["LAMBDA_BT"] if lam is None else lam
    z1, z2 = z1.float(), z2.float(); n = z1.shape[0]
    z1 = (z1 - z1.mean(0)) / (z1.std(0) + 1e-5); z2 = (z2 - z2.mean(0)) / (z2.std(0) + 1e-5)
    c = z1.T @ z2 / n; d = torch.diagonal(c)
    return (d - 1).pow(2).sum() + lam * (c.pow(2).sum() - d.pow(2).sum())

def perdida_vicreg(z1, z2):
    z1, z2 = z1.float(), z2.float(); n, d = z1.shape
    inv = F.mse_loss(z1, z2)
    std = lambda z: F.relu(1 - torch.sqrt(z.var(0) + 1e-4)).mean()
    def cov(z):
        z = z - z.mean(0); c = z.T @ z / max(n - 1, 1); return (c.pow(2).sum() - torch.diagonal(c).pow(2).sum()) / d
    return 25 * inv + 25 * (std(z1) + std(z2)) + cov(z1) + cov(z2)

def mascara_tramos(B, T, frac, dev, largo=10):
    m = torch.zeros(B, T, dtype=torch.bool, device=dev)
    n = max(1, int(T * frac / largo))
    ini = torch.randint(0, max(1, T - largo), (B, n), device=dev)
    for k in range(largo): m.scatter_(1, (ini + k).clamp(max=T - 1), True)
    return m

def subir(h, paso, T):
    return h.repeat_interleave(paso, dim=2)[:, :, :T] if paso > 1 else h[:, :, :T]

def ajustar_largo(h, T):
    return F.pad(h, (0, T - h.shape[2])) if h.shape[2] < T else h[:, :, :T]

## 9. Etapa 1: preentrenamiento de las capas bajas con ventanas públicas

Ventanas sueltas de HMOG y ExtraSensory, sin etiquetas. Se normaliza cada ventana con las estadísticas de su
sesión (equivale a la normalización por participante del dataset propio). Al final se comprueba que no colapsó
(varianza por dimensión) y se hace la prueba de fuga con HHAR: un clasificador lineal intenta predecir el modelo
de teléfono desde los vectores congelados, entrenando con unos usuarios y probando con otros.

In [ ]:
def ventanas_publicas():
    if not PUBLICOS: raise RuntimeError("No hay datos públicos para la etapa 1")
    W = np.concatenate([v[0] for v in PUBLICOS.values()]); Bd = np.concatenate([v[1] for v in PUBLICOS.values()])
    S = np.concatenate([v[2] for v in PUBLICOS.values()])
    return W, Bd, S

def stats_grupo(W, S):
    """Media y desvío por canal de cada sesión, para normalizar ventanas."""
    W32 = W.astype(np.float32); MU = np.zeros((len(W), C_DATOS), np.float32); SD = np.ones_like(MU)
    for s in np.unique(S):
        i = np.where(S == s)[0]; x = W32[i]
        mu = x.mean((0, 2)); sd = x.std((0, 2)).clip(1e-3)
        if CFG["NORM"] == "escala": mu = mu * 0
        MU[i], SD[i] = mu, sd
    return MU, SD

class ModeloE1(nn.Module):
    def __init__(s, tipo):
        super().__init__(); d = CFG["D"]
        s.bajas = Bajas(tipo, d=d); s.proy = proyector(d, CFG["PROY"]); s.dec = nn.Conv1d(d, C_DATOS, 1); s.gyr = nn.Conv1d(d, 3, 1)

def paso_e1(m, objetivo, x, band, mu, sd):
    B, _, T = x.shape; val = torch.ones(B, T, dtype=torch.bool, device=x.device)
    fases = torch.zeros(B, T, dtype=torch.int8, device=x.device); centro = torch.full((B,), T // 2, device=x.device)
    if objetivo in ("barlow", "vicreg"):
        v = []
        for _ in range(2):
            xa, _, va, ba = aumentar(x, fases, val, band, centro)
            h = m.bajas(preparar(xa, va, ba, mu, sd)); v.append(m.proy(h.mean(2)))
        return perdida_barlow(*v) if objetivo == "barlow" else perdida_vicreg(*v)
    xa, _, va, ba = aumentar(x, fases, val, band, centro)
    xn = preparar(xa, va, ba, mu, sd)
    if objetivo == "enmascarado":
        mk = mascara_tramos(B, T, 0.3 + 0.1 * random.random(), x.device)
        h = m.bajas(xn * (~mk)[:, None].float())
        rec = ajustar_largo(subir(m.dec(h), m.bajas.paso, T), T)
        peso = torch.ones(B, C_DATOS, T, device=x.device); peso[:, 3:6] *= ba[:, 0, None, None]; peso[:, 9] *= ba[:, 1, None]
        peso = peso * mk[:, None].float()
        return ((rec - xn[:, :C_DATOS]) ** 2 * peso).sum() / peso.sum().clamp(min=1)
    if objetivo == "entre_sensores":
        xin = xn.clone(); xin[:, 3:6] = 0; xin[:, 10] = 0
        h = m.bajas(xin); pred = ajustar_largo(subir(m.gyr(h), m.bajas.paso, T), T)
        peso = ba[:, 0, None, None]
        return ((pred - xn[:, 3:6]) ** 2 * peso).sum() / (peso.sum() * 3 * T).clamp(min=1)
    raise ValueError(objetivo)

def autocast():
    return torch.autocast("cuda", dtype=torch.bfloat16) if DEV.type == "cuda" else torch.autocast("cpu", enabled=False)

def entrenar_e1(tipo, objetivo, semilla):
    ck = RES / "ckpt" / f"e1_{tipo}_{objetivo}_s{semilla}.pt"
    m = ModeloE1(tipo).to(DEV)
    if ck.exists():
        m.bajas.load_state_dict(torch.load(ck, map_location=DEV)); return m.bajas
    fijar_semilla(semilla)
    W, Bd, S = ventanas_publicas(); MU, SD = stats_grupo(W, S)
    W = torch.from_numpy(W); Bd = torch.from_numpy(Bd); MU = torch.from_numpy(MU); SD = torch.from_numpy(SD)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    pasos = len(W) // CFG["E1_LOTE"]
    if CFG["E1_PASOS_MAX"]: pasos = min(pasos, CFG["E1_PASOS_MAX"])
    pasos = max(pasos, 1); total = pasos * CFG["E1_EPOCAS"]
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=total, pct_start=0.05)
    m.train(); t0 = time.time(); k = 0
    for ep in range(CFG["E1_EPOCAS"]):
        perm = torch.randperm(len(W)); acum = 0.0
        for p in range(pasos):
            idx = perm[p * CFG["E1_LOTE"]:(p + 1) * CFG["E1_LOTE"]]
            if len(idx) < 4: continue
            x = W[idx].float().to(DEV); b = Bd[idx].to(DEV); mu = MU[idx].to(DEV); sd = SD[idx].to(DEV)
            with autocast(): loss = paso_e1(m, objetivo, x, b, mu, sd)
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 3.0); opt.step(); sched.step()
            acum += loss.item(); k += 1
        if ep % max(1, CFG["E1_EPOCAS"] // 10) == 0 or ep == CFG["E1_EPOCAS"] - 1:
            print(f"  E1 {tipo}/{objetivo} s{semilla} época {ep + 1}/{CFG['E1_EPOCAS']} pérdida {acum / pasos:.4f} ({time.time() - t0:.0f} s)")
    torch.save(m.bajas.state_dict(), ck)
    # control de colapso
    m.eval()
    with torch.no_grad():
        idx = torch.randperm(len(W))[:2048]
        x = preparar(W[idx].float().to(DEV), torch.ones(len(idx), W.shape[2], dtype=torch.bool, device=DEV), Bd[idx].to(DEV), MU[idx].to(DEV), SD[idx].to(DEV))
        h = m.bajas(x).mean(2); sdv = h.std(0)
    info = dict(std_min=float(sdv.min()), std_media=float(sdv.mean()), dims_casi_cero=int((sdv < 1e-2).sum()))
    guardar_json(info, ck.with_suffix(".colapso.json"))
    print(f"  colapso: {info}")
    return m.bajas

def vectores_ventanas(bajas, W, Bd, MU, SD):
    bajas.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(W), 1024):
            x = torch.from_numpy(W[i:i + 1024].astype(np.float32)).to(DEV); n = len(x)
            xn = preparar(x, torch.ones(n, x.shape[2], dtype=torch.bool, device=DEV), torch.from_numpy(Bd[i:i + 1024]).to(DEV),
                          torch.from_numpy(MU[i:i + 1024]).to(DEV), torch.from_numpy(SD[i:i + 1024]).to(DEV))
            out.append(bajas(xn).mean(2).float().cpu().numpy())
    return np.concatenate(out)

def sonda_lineal(Z, y, grupos, semilla=0):
    """Exactitud balanceada de una regresión logística que predice y, entrenada y probada con grupos distintos."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import balanced_accuracy_score
    g = np.unique(grupos); rng = np.random.default_rng(semilla); rng.shuffle(g)
    prueba = set(g[: max(1, len(g) // 3)]); te = np.array([x in prueba for x in grupos])
    if te.all() or (~te).all() or len(np.unique(y[~te])) < 2: return float("nan"), 1 / max(len(np.unique(y)), 1)
    sc = StandardScaler().fit(Z[~te])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(sc.transform(Z[~te]), y[~te])
    return float(balanced_accuracy_score(y[te], clf.predict(sc.transform(Z[te])))), 1 / len(np.unique(y))

def prueba_fuga_hhar(bajas):
    if HHAR is None: return dict(fuga_hhar=float("nan"))
    W, Bd, MOD, USR = HHAR
    grupo = np.array([f"{a}|{b}" for a, b in zip(MOD, USR)]); MU, SD = stats_grupo(W, grupo)
    Z = vectores_ventanas(bajas, W, Bd, MU, SD)
    acc, azar = sonda_lineal(Z, MOD, USR)
    return dict(fuga_hhar=acc, fuga_hhar_azar=azar)

## 10. Etapa 2: ramas por fase y fusión con el dataset propio, sin identidad

Solo personas de entrenamiento. Cada lote es de **un solo participante**, así la huella del sensor no ayuda a
resolver ningún objetivo. Objetivos: el autosupervisado sobre el episodio completo (Barlow Twins o VICReg sobre el
vector del episodio), y tareas auxiliares con etiquetas que genera la app: origen de la toma (desde la rama de
toma, porque la fusión recibe el origen como contexto), fase de cada instante (ubicar fronteras) y episodio real
frente a falso disparo. El decodificador (autocodificador) y el predictor (uso a partir de toma y asentamiento) se
entrenan aquí sobre vectores desacoplados, para que no cambien el codificador. Dropout de fase: a veces se apaga
una rama entera, lo que permite medir cada fase en la ronda 5 sin reentrenar. El uso se corta en un x al azar
para que el modelo sirva en toda la curva EER frente a x.

In [ ]:
I_UP = CFG["PRE_S"] * FS
T_RED_DIV = 10

def objetivo_dec(xn, val):
    """Objetivo del autocodificador: episodio normalizado promediado de a 10 muestras."""
    T = xn.shape[2] // T_RED_DIV * T_RED_DIV
    x = F.avg_pool1d(xn[:, :C_DATOS, :T], T_RED_DIV); v = F.avg_pool1d(val[:, None, :T].float(), T_RED_DIV)[:, 0] > 0.5
    return x, v

def error_dec(cab, z, xn, val):
    obj, v = objetivo_dec(xn, val)
    rec = cab.dec(z).view(len(z), C_DATOS, -1)[:, :, :obj.shape[2]]
    w = v[:, None].float()
    return ((rec - obj) ** 2 * w).sum((1, 2)) / (w.sum((1, 2)) * C_DATOS).clamp(min=1)

def error_pred(cab, V, validas):
    p = cab.pred(torch.cat([V[:, 0], V[:, 1]], 1))
    e = 1 - F.cosine_similarity(p, V[:, 2], dim=1)
    return e, validas[:, 2] & validas[:, 0]

def lote_dispositivo(cj, idx):
    return (cj.X[idx].to(DEV, non_blocking=True), cj.val[idx].to(DEV), cj.fases[idx].to(DEV), cj.band[idx].to(DEV),
            cj.origen[idx].to(DEV), cj.i_toma[idx].to(DEV))

def ramas_al_azar(B, p, dev):
    on = torch.rand(B, 3, device=dev) >= p
    nada = ~on.any(1); on[nada, torch.randint(0, 3, (int(nada.sum()),), device=dev)] = True
    return on

def construir_modelo(tipo):
    m = ModeloEpisodio(Bajas(tipo, d=CFG["D"]), CFG["D"], CFG["DZ"]).to(DEV)
    T = (CFG["PRE_S"] + CFG["POST_S"]) * FS
    cab = Cabezas(CFG["DZ"], CFG["D"], T // T_RED_DIV).to(DEV)
    return m, cab

def entrenar_e2(tipo, pre, objetivo, semilla, cj, sujetos, bajas_modo=None, etiqueta=""):
    """pre: 'A' (con etapa 1 sobre públicos) o 'B' (sin públicos, desde cero). Devuelve (modelo, cabezas)."""
    bajas_modo = bajas_modo or CFG["E2_BAJAS"]
    if pre == "B": bajas_modo = "libre"
    nombre = f"e2_{tipo}_{pre}_{objetivo}_{bajas_modo}_{cj.nombre}{etiqueta}_s{semilla}.pt"
    ck = RES / "ckpt" / nombre
    m, cab = construir_modelo(tipo)
    if ck.exists():
        sd = torch.load(ck, map_location=DEV); m.load_state_dict(sd["m"]); cab.load_state_dict(sd["cab"]); return m, cab
    if pre == "A": m.bajas.load_state_dict(entrenar_e1(tipo, objetivo, semilla).state_dict())
    fijar_semilla(semilla)
    reales = {s: indices(cj, s) for s in sujetos}; reales = {s: i for s, i in reales.items() if len(i) >= 8}
    falsos = {s: indices(cj, s, tipos=("false_trigger",)) for s in reales}
    stats = {s: tuple(t.to(DEV) for t in stats_persona(cj, i)) for s, i in reales.items()}
    lr_b = {"congelar": 0.0, "lento": 1e-4, "libre": 1e-3}[bajas_modo]
    for p in m.bajas.parameters(): p.requires_grad = lr_b > 0
    for p in list(m.adap_bajas.parameters()) + [q for r in m.ramas for q in r.adap.parameters()]: p.requires_grad = False
    grupos = [dict(params=[p for n, p in m.named_parameters() if p.requires_grad and not n.startswith("bajas.")], lr=1e-3),
              dict(params=list(cab.parameters()), lr=1e-3)]
    if lr_b > 0: grupos.append(dict(params=list(m.bajas.parameters()), lr=lr_b))
    opt = torch.optim.AdamW(grupos, weight_decay=1e-4)
    suj = list(reales); peso = np.array([len(reales[s]) for s in suj], float); peso /= peso.sum()
    w = CFG["PESOS_AUX"]; t0 = time.time()
    for ep in range(CFG["E2_EPOCAS"]):
        m.train(); cab.train()
        if lr_b == 0: m.bajas.eval()
        acum = Counter()
        for paso in range(CFG["E2_PASOS_POR_EPOCA"]):
            s = suj[np.random.choice(len(suj), p=peso)]; mu, sd = stats[s]
            idx = np.random.choice(reales[s], min(CFG["E2_LOTE"], len(reales[s])), replace=len(reales[s]) < 4)
            x, val, fa, band, org, it = lote_dispositivo(cj, idx); B = len(idx)
            xs = torch.where(torch.rand(B, device=DEV) < 0.5, torch.rand(B, device=DEV) * CFG["X_MAX"], torch.full((B,), float(CFG["X_MAX"]), device=DEV))
            fa, val = truncar_x(fa, val, I_UP, xs, FS)
            centro = torch.full((B,), I_UP, device=DEV)
            outs, xns = [], []
            with autocast():
                for _ in range(2):
                    xa, fa_a, va, ba = aumentar(x, fa, val, band, centro)
                    xn = preparar(xa, va, ba, mu, sd); xns.append((xn, va))
                    outs.append(m(xn, fa_a, org, ramas_on=ramas_al_azar(B, CFG["P_DROPOUT_FASE"], DEV), i_toma=it))
                z1, z2 = cab.proy(outs[0]["z"]), cab.proy(outs[1]["z"])
                L_ssl = perdida_vicreg(z1, z2) if objetivo == "vicreg" else perdida_barlow(z1, z2)
                o = outs[0]
                lf = cab_f = m.cab_fases(o["h"]).float()
                L_fases = F.cross_entropy(lf, o["f"].long().clamp(min=-1), ignore_index=-1)
                tiene_o = org < 3
                L_origen = F.cross_entropy(m.cab_origen(o["V"][:, 0]).float()[tiene_o], org[tiene_o]) if tiene_o.any() else x.new_zeros(())
                logit_real = [o["real"].float()]; y_real = [torch.ones(B, dtype=torch.long, device=DEV)]
                if len(falsos[s]):
                    jf = np.random.choice(falsos[s], min(max(B // 4, 2), len(falsos[s])), replace=False)
                    xf, vf, ff, bf, of, itf = lote_dispositivo(cj, jf)
                    xfa, ffa, vfa, bfa = aumentar(xf, ff, vf, bf, torch.full((len(jf),), I_UP, device=DEV))
                    of_ = m(preparar(xfa, vfa, bfa, mu, sd), ffa, of, i_toma=itf)
                    logit_real.append(of_["real"].float()); y_real.append(torch.zeros(len(jf), dtype=torch.long, device=DEV))
                L_real = F.cross_entropy(torch.cat(logit_real), torch.cat(y_real))
                L_dec = error_dec(cab, o["z"].detach(), *xns[0]).mean()
                e_p, ok_p = error_pred(cab, o["V"].detach(), o["validas"])
                L_pred = e_p[ok_p].mean() if ok_p.any() else x.new_zeros(())
                loss = L_ssl + w["origen"] * L_origen + w["fases"] * L_fases + w["real"] * L_real + L_dec + L_pred
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_([p for g in grupos for p in g["params"]], 3.0); opt.step()
            for k, v in dict(ssl=L_ssl, origen=L_origen, fases=L_fases, real=L_real, dec=L_dec, pred=L_pred).items(): acum[k] += float(v)
        if ep % max(1, CFG["E2_EPOCAS"] // 10) == 0 or ep == CFG["E2_EPOCAS"] - 1:
            n = CFG["E2_PASOS_POR_EPOCA"]
            print(f"  E2 {tipo}/{pre}/{objetivo} s{semilla} época {ep + 1}: " + " ".join(f"{k} {v / n:.3f}" for k, v in acum.items()) + f" ({time.time() - t0:.0f} s)")
    torch.save(dict(m=m.state_dict(), cab=cab.state_dict()), ck)
    return m, cab

def aux_validacion(m, cj, sujetos):
    """Exactitud de las tareas auxiliares en personas de validación (deben superar claramente al azar)."""
    m.eval(); ac = Counter(); n = Counter()
    with torch.no_grad():
        for s in sujetos:
            i = indices(cj, s)
            if len(i) == 0: continue
            mu, sd = (t.to(DEV) for t in stats_persona(cj, i))
            for a in range(0, len(i), 64):
                x, val, fa, band, org, it = lote_dispositivo(cj, i[a:a + 64])
                o = m(preparar(x, val, band, mu, sd), fa, org, i_toma=it)
                pf = m.cab_fases(o["h"]).argmax(1); ok = o["f"] >= 0
                ac["fases"] += float((pf == o["f"].long())[ok].sum()); n["fases"] += float(ok.sum())
                t = org < 3
                if t.any(): ac["origen"] += float((m.cab_origen(o["V"][:, 0]).argmax(1) == org)[t].sum()); n["origen"] += float(t.sum())
                ac["real"] += float((o["real"].argmax(1) == 1).sum()); n["real"] += len(x)
            j = indices(cj, s, tipos=("false_trigger",))
            for a in range(0, len(j), 64):
                x, val, fa, band, org, it = lote_dispositivo(cj, j[a:a + 64])
                o = m(preparar(x, val, band, mu, sd), fa, org, i_toma=it)
                ac["real"] += float((o["real"].argmax(1) == 0).sum()); n["real"] += len(x)
    return {f"aux_{k}": ac[k] / n[k] for k in n if n[k]}

## 11. Etapa 3: aprendizaje del dueño y evaluación

Cada persona de validación o prueba se trata como un usuario nuevo: recibe una copia del modelo de la etapa 2 y
solo conoce sus propios episodios. En orden temporal: enrolamiento (los primeros `ENROLAR_N`; en la ronda 3,
además, los de la semana 1, y de forma provisional los del día 1 y los días 1–3), calibración del umbral (los `CALIBRAR_N` siguientes; percentil 5
de sus puntajes, FRR objetivo ≈ 5 %) y evaluación con el resto.

**Impostores.** Mismo teléfono: episodios con `impostor_session = true` en el teléfono del dueño (sesiones de
laboratorio). Otros teléfonos: episodios de evaluación de las demás personas del mismo grupo. Todos pasan por la
normalización del dueño, como pasaría en su teléfono.

**Esquemas.** `conjunto` (codificador fijo; similitud coseno al centroide del mismo origen, o al global si hay
menos de 5), `svdd` (cabeza de una clase con centro fijo), `ae` (error de reconstrucción), `pred` (error al
predecir el vector del uso desde la toma y el asentamiento; no existe con x = 0) e `hibrido` (combina `conjunto` y
`HIBRIDO_SEGUNDO`: cada uno se ajusta en su propia copia del modelo, sus puntajes se llevan a media 0 y desvío 1
con los episodios de calibración del dueño y se promedian).

**Niveles de ajuste.** `nada`, `normalizacion` (se reestiman las estadísticas de normalización por lotes con el
dueño), `cabeza`, `adaptadores`, `ultimas` (ramas, fusión y salida) y `todo`. Lo que se entrena usa la pérdida
del esquema; en `conjunto`, Barlow Twins sobre el dueño con su cabeza lineal.

In [ ]:
import copy
NIVEL_PARAMS = {
    "adaptadores": lambda m: list(m.adap_bajas.parameters()) + [p for r in m.ramas for p in r.adap.parameters()],
    "ultimas": lambda m: [p for n, p in m.named_parameters() if n.startswith(("ramas.", "fusion.", "salida.", "emb_"))],
    "todo": lambda m: list(m.parameters()),
}
CABEZA_ESQ = {"conjunto": "lineal", "svdd": "svdd", "ae": "dec", "pred": "pred"}

def adaptar_bn(m, cj, idx, mu, sd):
    for mod in m.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm): mod.reset_running_stats(); mod.momentum = None
    m.train()
    with torch.no_grad():
        for a in range(0, len(idx), 64):
            x, val, fa, band, org, it = lote_dispositivo(cj, idx[a:a + 64])
            m(preparar(x, val, band, mu, sd), fa, org)
    m.eval()

def salidas(m, cab, cj, idx, mu, sd, x_s, ramas_on=None):
    """z, V, validas y error de reconstrucción para episodios idx, con el uso cortado en x_s segundos."""
    m.eval(); cab.eval(); Z, V, VAL, ER = [], [], [], []
    with torch.no_grad():
        for a in range(0, len(idx), 128):
            x, val, fa, band, org, it = lote_dispositivo(cj, idx[a:a + 128])
            fa, val = truncar_x(fa, val, I_UP, x_s, FS)
            xn = preparar(x, val, band, mu, sd)
            ro = None if ramas_on is None else ramas_on.to(DEV).expand(len(x), 3)
            o = m(xn, fa, org, ramas_on=ro)
            Z.append(o["z"].float()); V.append(o["V"].float()); VAL.append(o["validas"]); ER.append(error_dec(cab, o["z"], xn, val).float())
    cat = lambda l: torch.cat(l) if l else torch.zeros(0, device=DEV)
    return dict(z=cat(Z), V=cat(V), validas=cat(VAL), err_dec=cat(ER))

@torch.no_grad()
def puntajes(esq, cab, ref, s, origen_ref, origen_s):
    """Puntaje de parecido con el dueño (más alto = más dueño)."""
    if esq == "conjunto":
        zr = F.normalize(cab.lineal(ref["z"]), dim=1); zs = F.normalize(cab.lineal(s["z"]), dim=1)
        g = F.normalize(zr.mean(0, keepdim=True), dim=1)
        out = (zs @ g.T)[:, 0]
        for o in np.unique(origen_s):
            r = origen_ref == o
            if r.sum() >= 5:
                c = F.normalize(zr[torch.as_tensor(r, device=DEV)].mean(0, keepdim=True), dim=1)
                sel = torch.as_tensor(origen_s == o, device=DEV); out[sel] = (zs[sel] @ c.T)[:, 0]
        return out.cpu().numpy()
    if esq == "svdd":
        return -((cab.svdd(s["z"]) - cab.centro) ** 2).sum(1).cpu().numpy()
    if esq == "ae":
        return -s["err_dec"].cpu().numpy()
    if esq == "pred":
        e, ok = error_pred(cab, s["V"], s["validas"])
        r = -e.cpu().numpy(); r[~ok.cpu().numpy()] = np.nan; return r
    raise ValueError(esq)

def esquemas_de(esq): return ["conjunto", CFG["HIBRIDO_SEGUNDO"]] if esq == "hibrido" else [esq]

def ajustar_esquemas(m0, cab0, cj, enr, mu, sd, esq, nivel, semilla):
    """Una copia del modelo ajustada al dueño por cada esquema que interviene (dos en el híbrido)."""
    out = []
    for e in esquemas_de(esq):
        m, cab = copy.deepcopy(m0), copy.deepcopy(cab0)
        ajustar_dueno(m, cab, cj, enr, mu, sd, e, nivel, semilla); out.append((e, m, cab))
    return out

def puntuar(modelos, cj, enr, grupos, mu, sd, x_s, ramas_on=None, ref_norm="cal"):
    """Puntajes de cada grupo de episodios. Con varios esquemas (híbrido), cada uno se lleva a media 0 y desvío 1
    con sus puntajes en ref_norm (la calibración del dueño; si no hay, el enrolamiento) y se promedian."""
    org = cj.meta.origen.to_numpy()
    if len(modelos) > 1 and ref_norm not in grupos: grupos = {**grupos, "_norm": enr}; ref_norm = "_norm"
    crudos = []
    for e, m, cab in modelos:
        ref = salidas(m, cab, cj, enr, mu, sd, x_s, ramas_on)
        crudos.append({k: puntajes(e, cab, ref, salidas(m, cab, cj, v, mu, sd, x_s, ramas_on), org[enr], org[v]) if len(v) else np.zeros(0)
                       for k, v in grupos.items()})
    if len(crudos) == 1: return crudos[0]
    norma = [(np.nanmean(c[ref_norm]), np.nanstd(c[ref_norm]) + 1e-6) for c in crudos]
    return {k: np.mean([(c[k] - a) / b for c, (a, b) in zip(crudos, norma)], 0) if len(v) else np.zeros(0) for k, v in grupos.items()}

def ajustar_dueno(m, cab, cj, enr, mu, sd, esq, nivel, semilla):
    fijar_semilla(semilla)
    for p in list(m.parameters()) + list(cab.parameters()): p.requires_grad = False
    if nivel == "normalizacion": adaptar_bn(m, cj, enr, mu, sd)
    if esq == "svdd":
        with torch.no_grad():
            z = salidas(m, cab, cj, enr, mu, sd, CFG["X_MAX"])["z"]
            c = cab.svdd(z).mean(0); c[(c.abs() < 0.01)] = 0.01      # centro fijo, lejos del origen (evita el colapso trivial)
        cab.centro = c
    if nivel == "nada" and esq == "conjunto": return
    params = list(getattr(cab, CABEZA_ESQ[esq]).parameters())
    if nivel in NIVEL_PARAMS: params += NIVEL_PARAMS[nivel](m)
    for p in params: p.requires_grad = True
    lr = 1e-3 if nivel in ("nada", "normalizacion", "cabeza") else 1e-4
    opt = torch.optim.AdamW([{"params": [p for p in params if any(p is q for q in cab.parameters())], "lr": 1e-3},
                             {"params": [p for p in params if not any(p is q for q in cab.parameters())], "lr": lr}], weight_decay=1e-4)
    m.eval(); cab.eval()          # estadísticas de normalización fijas (salvo el nivel "normalizacion", ya ajustado)
    for ep in range(CFG["E3_EPOCAS"]):
        perm = np.random.permutation(enr)
        for a in range(0, len(perm), CFG["E3_LOTE"]):
            idx = perm[a:a + CFG["E3_LOTE"]]
            if len(idx) < 4: continue
            x, val, fa, band, org, it = lote_dispositivo(cj, idx); B = len(idx)
            with autocast():
                xa, fa_a, va, ba = aumentar(x, fa, val, band, torch.full((B,), I_UP, device=DEV))
                xn = preparar(xa, va, ba, mu, sd); o = m(xn, fa_a, org)
                if esq == "conjunto":
                    xb, fb, vb, bb = aumentar(x, fa, val, band, torch.full((B,), I_UP, device=DEV))
                    o2 = m(preparar(xb, vb, bb, mu, sd), fb, org)
                    loss = perdida_barlow(cab.lineal(o["z"]), cab.lineal(o2["z"]))
                elif esq == "svdd": loss = ((cab.svdd(o["z"]) - cab.centro) ** 2).sum(1).mean()
                elif esq == "ae": loss = error_dec(cab, o["z"], xn, va).mean()
                elif esq == "pred":
                    e, ok = error_pred(cab, o["V"], o["validas"]); loss = e[ok].mean() if ok.any() else e.sum() * 0
            opt.zero_grad(set_to_none=True); loss.float().backward(); opt.step()

def eer(gen, imp):
    gen = np.asarray(gen, float); imp = np.asarray(imp, float)
    gen, imp = gen[np.isfinite(gen)], imp[np.isfinite(imp)]
    if len(gen) < 3 or len(imp) < 3: return float("nan")
    y = np.r_[np.ones(len(gen)), np.zeros(len(imp))]; s = np.r_[gen, imp]
    fpr, tpr, _ = roc_curve(y, s); fnr = 1 - tpr; i = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[i] + fnr[i]) / 2 * 100)

def far_en(imp, umbral):
    imp = np.asarray(imp, float); imp = imp[np.isfinite(imp)]
    return float((imp >= umbral).mean() * 100) if len(imp) else float("nan")

def division_dueno(cj, s, enrol="N"):
    """Índices en orden temporal: enrolamiento, calibración y evaluación del dueño s."""
    i = orden_temporal(cj, indices(cj, s))
    if enrol == "N":
        a, b = CFG["ENROLAR_N"], CFG["ENROLAR_N"] + CFG["CALIBRAR_N"]
        return i[:a], i[a:b], i[b:]
    # "semanaN": las primeras N semanas; "diasN": las primeras N × 24 h desde el primer episodio
    dias = 7 * int(enrol[len("semana"):]) if enrol.startswith("semana") else int(enrol[len("dias"):])
    f = cj.meta.fecha.to_numpy()[i]
    corte = f[0] + np.timedelta64(dias, "D"); dentro = i[f < corte]
    if len(i) - len(dentro) < 10 or len(dentro) < CFG["ENROLAR_N"] // 2 + CFG["CALIBRAR_N"]: return None
    return dentro[:-CFG["CALIBRAR_N"]], dentro[-CFG["CALIBRAR_N"]:], i[f >= corte]

def evaluar_persona(m0, cab0, cj, s, otros, esq, nivel, semilla, enrol="N", ramas_on=None, x_lista=None):
    div = division_dueno(cj, s, enrol)
    if div is None: return None
    enr, cal, ev = div
    if len(ev) < 5: return None
    mu, sd = (t.to(DEV) for t in stats_persona(cj, enr))
    modelos = ajustar_esquemas(m0, cab0, cj, enr, mu, sd, esq, nivel, semilla)
    esq_ef = "+".join(esquemas_de(esq))
    imp_m = indices(cj, s, impostor=True)
    imp_o = impostores_otros(cj, s, otros)
    org = cj.meta.origen.to_numpy()
    x_lista = x_lista if x_lista is not None else list(range(0, CFG["X_MAX"] + 1))
    res = dict(sujeto=s, n_enrol=len(enr), n_eval=len(ev), n_imp_mismo=len(imp_m), n_imp_otros=len(imp_o), esquema_efectivo=esq_ef)
    for x_s in x_lista:
        sc = puntuar(modelos, cj, enr, dict(ev=ev, cal=cal, im=imp_m, io=imp_o), mu, sd, x_s, ramas_on)
        g, c_, im, io_ = sc["ev"], sc["cal"], sc["im"], sc["io"]
        umbral = np.nanpercentile(c_, CFG["FRR_OBJETIVO"] * 100) if np.isfinite(c_).any() else np.nan
        res[f"eer_mismo@{x_s}"] = eer(g, im); res[f"eer_otros@{x_s}"] = eer(g, io_)
        res[f"far_mismo@{x_s}"] = far_en(im, umbral); res[f"far_otros@{x_s}"] = far_en(io_, umbral)
        res[f"frr@{x_s}"] = float((g[np.isfinite(g)] < umbral).mean() * 100) if np.isfinite(g).any() else np.nan
        if x_s == x_lista[-1]:
            for o in np.unique(org[ev]):
                sel = org[ev] == o
                if sel.sum() >= 5: res[f"eer_otros_origen_{o}"] = eer(g[sel], io_)
            sem = ((cj.meta.fecha.to_numpy()[ev] - cj.meta.fecha.to_numpy()[enr][0]) / np.timedelta64(7, "D")).astype(int)
            for w in np.unique(sem):
                if (sem == w).sum() >= 5: res[f"eer_otros_semana_{w + 1}"] = eer(g[sem == w], io_)
            if esq_ef == "conjunto": res.update(actualizacion(modelos[0][1], modelos[0][2], cj, enr, ev, imp_m, imp_o, mu, sd, umbral, x_s))
    return res

def actualizacion(m, cab, cj, enr, ev, imp_m, imp_o, mu, sd, umbral, x_s):
    """Agrega al perfil los episodios del flujo con puntaje alto (con tope diario) y mide el EER por semana.
    El flujo del teléfono del dueño incluye a los impostores del mismo teléfono en su fecha: si pasan, contaminan."""
    org = cj.meta.origen.to_numpy(); fecha = cj.meta.fecha.to_numpy()
    flujo = np.concatenate([ev, imp_m]); flujo = flujo[np.argsort(fecha[flujo], kind="stable")]
    es_imp = np.isin(flujo, imp_m)
    perfil = list(enr); agregados = 0; contaminados = 0; out = {}
    dias = pd.to_datetime(fecha[flujo]).normalize()
    for d in np.unique(dias):
        hoy = flujo[dias == d]; imp_hoy = es_imp[dias == d]
        ref = salidas(m, cab, cj, np.array(perfil), mu, sd, x_s)
        s = puntajes("conjunto", cab, ref, salidas(m, cab, cj, hoy, mu, sd, x_s), org[np.array(perfil)], org[hoy])
        alto = np.argsort(-s)[: CFG["ACTUALIZAR_POR_DIA"]]
        alto = [k for k in alto if s[k] >= umbral]
        perfil += list(hoy[alto]); agregados += len(alto); contaminados += int(imp_hoy[alto].sum())
    ref = salidas(m, cab, cj, np.array(perfil), mu, sd, x_s)
    g = puntajes("conjunto", cab, ref, salidas(m, cab, cj, ev, mu, sd, x_s), org[np.array(perfil)], org[ev])
    io_ = puntajes("conjunto", cab, ref, salidas(m, cab, cj, imp_o, mu, sd, x_s), org[np.array(perfil)], org[imp_o]) if len(imp_o) else np.zeros(0)
    out["eer_otros_con_actualizacion"] = eer(g, io_); out["actualizados"] = agregados; out["impostores_agregados"] = contaminados
    return out

## 12. Tabla de seguimiento y ejecutor de filas

Cada fila de la tabla del `.md` se corre con `correr_fila`, que entrena (o recupera de `ckpt/`) las etapas 1 y 2
para cada semilla y evalúa a cada persona del grupo. Se guarda una fila por prueba en
`tabla_seguimiento.csv` y el detalle por persona en `detalle/`. Si la fila ya está, no se vuelve a correr.

**Métrica que decide.** EER con impostores en el mismo teléfono, promediado por persona. Mientras no haya
sesiones de impostor, se usa el EER frente a otros teléfonos y la tabla lo anota. Si dos opciones quedan a menos de
1 punto, se elige la más simple (menos parámetros, o el primer objetivo de la lista).

In [ ]:
TABLA = RES / "tabla_seguimiento.csv"; (RES / "detalle").mkdir(exist_ok=True)

def leer_tabla(): return pd.read_csv(TABLA, dtype={"ID": str}) if TABLA.exists() else pd.DataFrame(columns=["ID"])

def agregar_fila(fila):
    t = leer_tabla(); t = t[t.ID != fila["ID"]] if len(t) else t
    t = pd.concat([t, pd.DataFrame([fila])], ignore_index=True); t.to_csv(TABLA, index=False); return t

def sujetos_de(grupo, cj=None):
    cj = cj or PROPIO
    presentes = set(cj.meta.sujeto)
    return [s for s, g in PART.items() if g == grupo and s in presentes]

def correr_fila(ID, arq, pre="A", obj="barlow", esquema="conjunto", nivel="nada", grupo="validacion", enrol="N",
                ramas=None, bajas_modo=None, cj=None, notas="", forzar=False, x_lista=None):
    t = leer_tabla()
    if not forzar and len(t) and ID in set(t.ID): return t[t.ID == ID].iloc[0].to_dict()
    cj = cj or PROPIO
    ent, evals = sujetos_de("entrenamiento", cj), sujetos_de(grupo, cj)
    ramas_on = None if ramas is None else torch.tensor([[r in ramas for r in ("toma", "asentamiento", "uso")]])
    filas, extra = [], {}
    for sem in CFG["SEMILLAS"]:
        if pre == "cero":
            m, cab = construir_modelo(arq); nivel_ef = "todo"
        else:
            m, cab = entrenar_e2(arq, pre, obj, sem, cj, ent, bajas_modo=bajas_modo); nivel_ef = nivel
            if pre == "A" and not extra:
                extra.update({k + "_etapa1": v for k, v in prueba_fuga_hhar(entrenar_e1(arq, obj, sem)).items()})
                extra.update(prueba_fuga_hhar(m.bajas)); extra.update(aux_validacion(m, cj, evals))
        for s in evals:
            r = evaluar_persona(m, cab, cj, s, evals, esquema, nivel_ef, sem, enrol=enrol, ramas_on=ramas_on, x_lista=x_lista)
            if r is not None: r["semilla"] = sem; filas.append(r)
    det = pd.DataFrame(filas); det.to_csv(RES / "detalle" / f"{ID}.csv", index=False)
    if det.empty: print(ID, "sin personas evaluables"); return {}
    x_max = (x_lista or [CFG["X_MAX"]])[-1]
    por_sem = det.groupby("semilla")[[c for c in det.columns if "@" in c or c.startswith("eer_")]].mean(numeric_only=True)
    m_ = por_sem.mean(); s_ = por_sem.std().fillna(0)
    hay_mismo = det[f"eer_mismo@{x_max}"].notna().any()
    fila = dict(ID=ID, Arquitectura=arq, Preentrenamiento=pre, Objetivo=obj, Esquema=esquema, Ajuste=nivel,
                Grupo=grupo, Enrolamiento=enrol, Ramas=",".join(ramas) if ramas else "todas",
                EER_mismo=m_.get(f"eer_mismo@{x_max}"), EER_mismo_sd=s_.get(f"eer_mismo@{x_max}"),
                EER_otros=m_.get(f"eer_otros@{x_max}"), EER_otros_sd=s_.get(f"eer_otros@{x_max}"),
                FAR_mismo_FRR5=m_.get(f"far_mismo@{x_max}"), FAR_otros_FRR5=m_.get(f"far_otros@{x_max}"), FRR_real=m_.get(f"frr@{x_max}"),
                Metrica_decision="EER_mismo" if hay_mismo else "EER_otros (no hay impostores en el mismo teléfono)",
                Personas=det.sujeto.nunique(), Semillas=len(CFG["SEMILLAS"]),
                Parametros=n_param(construir_modelo(arq)[0]), Estado="Hecho", Notas=notas, **extra)
    fila["Brecha"] = (fila["EER_mismo"] - fila["EER_otros"]) if hay_mismo else np.nan
    for x_s in (x_lista or range(0, CFG["X_MAX"] + 1)):
        fila[f"EER_otros@{x_s}s"] = m_.get(f"eer_otros@{x_s}"); fila[f"EER_mismo@{x_s}s"] = m_.get(f"eer_mismo@{x_s}")
    agregar_fila(fila)
    print(f"{ID}: EER otros {fila['EER_otros']:.2f} ± {fila['EER_otros_sd']:.2f} | EER mismo {fila['EER_mismo']} | {notas}")
    return fila

def decision(fila): return fila["EER_mismo"] if pd.notna(fila.get("EER_mismo")) else fila["EER_otros"]

def mejor(ids, clave_simple):
    """La de menor EER; si otra queda a menos de 1 punto y es más simple, gana la más simple."""
    t = leer_tabla(); t = t[t.ID.isin(ids)].copy()
    t["dec"] = t.apply(decision, axis=1); t = t[t.dec.notna()]
    if t.empty: raise RuntimeError(f"Sin resultados para {ids}")
    b = t.dec.min(); cand = t[t.dec <= b + CFG["UMBRAL_EMPATE"]]
    return cand.sort_values(clave_simple).iloc[0].to_dict()

## 13. Líneas base de trabajos previos (camino 1: sus métodos en nuestras condiciones reales)

Mismas personas, mismo enrolamiento, misma calibración del umbral y mismos impostores que nuestro modelo. Ningún
artículo publicó código (se buscó en los artículos, las páginas de los autores y GitHub), así que cada método se
implementa según su texto. Lo que el artículo no dice, o lo que nuestros datos no permiten, se anota como diferencia.

**Secure Pick Up** (Lee et al., 2017)
- Señal: la toma desde el fin del reposo hasta que se enciende la pantalla (sin el segundo previo que agrega nuestra
  toma), hasta 4 s, acelerómetro (m/s²) y giroscopio (rad/s).
- Distancia: DTW por eje, sin banda, sumado con pesos 0,6 para el acelerómetro y 0,4 para el giroscopio, repartidos
  por igual entre ejes (ec. 3).
- Perfil del dueño: una sola toma promedio. Cada toma de enrolamiento se alinea con el perfil por DTW y se promedia
  con él (sección 4.3). Puntaje: menos la distancia al perfil.
- Solo decide en tomas desde una superficie (el artículo solo autentica tomas que parten de un reposo, ~36 % de los
  desbloqueos; las demás piden la autenticación explícita). La tabla anota esa **cobertura**.
- Diferencias: el umbral se calibra por persona (el artículo usa uno global, θ = 2,8) y no se actualiza el perfil
  durante la evaluación, igual que con los demás métodos.

**Please Hold On** (Buriro et al., 2017)
- Señal: los x s después del desbloqueo, a 5 Hz (SENSOR_DELAY_NORMAL, una muestra cada 200 ms).
- Rasgos: 7 estadísticos (media, desviación absoluta media, mediana, error estándar de la media, desvío, asimetría
  sin sesgo y curtosis) de los 3 ejes y la magnitud de: acelerómetro, gravedad, giroscopio, y el acelerómetro con
  los filtros pasa bajos y pasa altos de Android (α = 0,5). Sin escalar.
- Selección: los 50 rasgos con más ganancia de información. Clasificador: Random Forest de 100 árboles con los
  parámetros por defecto de WEKA (log2(rasgos) + 1 rasgos por división), dueño contra otras personas.
- Diferencias: el magnetómetro entra solo como magnitud (nuestros datos guardan |mag| relativa) y falta el sensor de
  orientación; WEKA discretiza con MDL antes de medir la ganancia y aquí se usan 10 intervalos de igual frecuencia.

**Boshoff et al. (2023)**
- Señal: 7 s desde el inicio de la toma (en el artículo, 7 s desde el bolsillo), acelerómetro, giroscopio y gravedad,
  interpolada (cúbica) a 1 kHz y filtrada con un Butterworth pasa bajos de orden 10 y corte en 7 Hz.
- Rasgos: cada flujo a media 0 y varianza 1; un solo vocabulario de d = 90 "palabras de movimiento" (k-means por
  minilotes sobre las muestras de todos los flujos juntos) e histograma por episodio.
- Clasificadores: MLP binario (dueño contra otras personas, el mejor del artículo) y uno de una clase. El artículo
  no dice cuál usó como clasificador de una clase; aquí es un OC-SVM.
- Diferencias: el artículo usa la matriz de rotación, que no grabamos; la gravedad la reemplaza como medida de la
  orientación.

**Si un episodio no tiene toma** (pasa en ~2 de cada 3 episodios de Motion ID, donde la pantalla se enciende al
empezar a levantar el teléfono, y en ~1 de cada 5 del dataset propio), Secure Pick Up usa todo el movimiento previo
al desbloqueo (toma + asentamiento).

In [ ]:
from scipy import stats as sst
try:
    from numba import njit
except ImportError:
    njit = lambda f=None, **k: (f if f else (lambda g: g))

@njit(cache=True)
def _dtw1(a, b, banda):
    n, m = len(a), len(b); INF = 1e18
    D = np.full((n + 1, m + 1), INF); D[0, 0] = 0.0
    for i in range(1, n + 1):
        j0 = max(1, i * m // n - banda); j1 = min(m, i * m // n + banda)
        for j in range(j0, j1 + 1):
            c = abs(a[i - 1] - b[j - 1])
            D[i, j] = c + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return D[n, m] / (n + m)

@njit(cache=True)
def _dtw_pond(A, B, pesos, banda):
    s = 0.0
    for k in range(A.shape[1]): s += pesos[k] * _dtw1(A[:, k], B[:, k], banda)
    return s

def tramo(cj, i, fase, x_s=None, max_largo=None):
    f = cj.fases[i].numpy(); X = cj.X[i].numpy()
    if fase == 3 and x_s is not None:
        f = f.copy(); f[I_UP + int(x_s * FS):] = -1
    sel = np.where(f == fase)[0]
    seg = X[:, sel].T
    if max_largo and len(seg) > max_largo: seg = seg[-max_largo:]
    return seg

def tramo_toma(cj, i, max_largo=None):
    """La toma; si el episodio no la tiene (en Motion ID la pantalla se enciende al empezar a levantar el teléfono, y
    casi todo el gesto cae en el asentamiento), todo el movimiento previo al desbloqueo: toma + asentamiento."""
    seg = tramo(cj, i, 1, max_largo=max_largo)
    if len(seg) >= 10: return seg
    f = cj.fases[i].numpy(); seg = cj.X[i].numpy()[:, np.where((f == 1) | (f == 2))[0]].T
    return seg[-max_largo:] if max_largo and len(seg) > max_largo else seg

@njit(cache=True)
def _dtw1_alinear(a, b):
    """DTW completo entre a y b; para cada muestra de a, el promedio de las muestras de b alineadas con ella."""
    n, m = len(a), len(b); INF = 1e18
    D = np.full((n + 1, m + 1), INF); D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = abs(a[i - 1] - b[j - 1]) + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    suma = np.zeros(n); cuenta = np.zeros(n); i, j = n, m
    while i > 0 and j > 0:
        suma[i - 1] += b[j - 1]; cuenta[i - 1] += 1
        d, u, l = D[i - 1, j - 1], D[i - 1, j], D[i, j - 1]
        if d <= u and d <= l: i -= 1; j -= 1
        elif u <= l: i -= 1
        else: j -= 1
    return suma / np.maximum(cuenta, 1.0)

# --- Secure Pick Up (Lee et al., 2017) ---------------------------------------------------------------------------
PESOS_SPU = np.array([0.2, 0.2, 0.2, 0.4 / 3, 0.4 / 3, 0.4 / 3])     # acelerómetro 0,6 y giroscopio 0,4, por igual entre ejes

def spu_senal(cj, i):
    """Señal de la toma como en SPU: desde el fin del reposo hasta que se enciende la pantalla (0,5 a 4 s), acelerómetro
    en m/s² y giroscopio en rad/s. Nuestra toma empieza 1 s antes del movimiento: ese segundo se quita."""
    seg = tramo(cj, i, 1)
    seg = seg[FS:] if len(seg) >= FS + 10 else tramo_toma(cj, i)
    seg = seg[-4 * FS:, :6].astype(np.float64)
    seg[:, :3] *= G0
    return seg

def spu_perfil(senales):
    """Perfil del dueño: se promedia el perfil guardado con cada toma nueva, alineadas con DTW por eje (sección 4.3)."""
    p = senales[0].copy()
    for x in senales[1:]:
        p = (p + np.stack([_dtw1_alinear(p[:, k], x[:, k]) for k in range(p.shape[1])], 1)) / 2
    return p

# --- Please Hold On (Buriro et al., 2017) --------------------------------------------------------------------------
def _filtro_android(a, alfa=0.5):
    """Pasa bajos de la documentación de Android (g = α·g + (1 − α)·a) y pasa altos (a − g)."""
    g = np.empty_like(a); g[0] = a[0]
    for t in range(1, len(a)): g[t] = alfa * g[t - 1] + (1 - alfa) * a[t]
    return g, a - g

def rasgos_buriro(cj, i, x_s):
    """7 estadísticos por eje y magnitud de cada sensor, sobre los x s después del desbloqueo, a 5 Hz."""
    seg = tramo(cj, i, 3, x_s=x_s)[:: max(1, FS // 5)]       # SENSOR_DELAY_NORMAL: una muestra cada 200 ms
    n = 7 * (5 * 4 + 1)
    if len(seg) < 5: return np.full(n, np.nan)
    acc, gyr, grav, mg = seg[:, 0:3], seg[:, 3:6], seg[:, 6:9], seg[:, 9:10]
    lpf, hpf = _filtro_android(acc)
    dims = np.concatenate([np.concatenate([s, np.linalg.norm(s, axis=1, keepdims=True)], 1)
                           for s in (acc, grav, gyr, lpf, hpf)] + [mg], 1)
    f = [dims.mean(0), np.abs(dims - dims.mean(0)).mean(0), np.median(dims, 0), sst.sem(dims, 0), dims.std(0, ddof=1),
         sst.skew(dims, 0, bias=False), sst.kurtosis(dims, 0)]
    return np.nan_to_num(np.concatenate(f))

def ganancia_informacion(F, y, bins=10):
    """Ganancia de información de cada rasgo respecto de la clase, con rasgos discretizados en intervalos de igual
    frecuencia (aproxima InfoGainAttributeEval de WEKA, que discretiza con MDL)."""
    def H(v):
        p = np.bincount(v) / len(v); p = p[p > 0]; return -(p * np.log2(p)).sum()
    y = y.astype(int); h = H(y); out = np.zeros(F.shape[1])
    for k in range(F.shape[1]):
        cortes = np.unique(np.quantile(F[:, k], np.linspace(0, 1, bins + 1)[1:-1]))
        b = np.digitize(F[:, k], cortes)
        out[k] = h - sum((b == v).mean() * H(y[b == v]) for v in np.unique(b))
    return out

# --- Boshoff et al. (2023) ----------------------------------------------------------------------------------------
def boshoff_senal(cj, i):
    """Ventana de 7 s desde la toma (o lo que haya), acelerómetro, giroscopio y gravedad, interpolada (cúbica) a 1 kHz
    y filtrada con un Butterworth pasa bajos de orden 10 y corte en 7 Hz, como en el artículo."""
    from scipy.interpolate import CubicSpline
    X = cj.X[i].numpy(); v = cj.val[i].numpy(); a = int(cj.i_toma[i])
    sel = np.where(v[a:a + 7 * FS])[0] + a
    if len(sel) < FS: return None
    seg = X[[0, 1, 2, 3, 4, 5, 6, 7, 8]][:, sel].T.astype(np.float64); seg[:, [0, 1, 2, 6, 7, 8]] *= G0
    t = sel / FS; tn = np.arange(t[0], t[-1], 0.001)
    seg = CubicSpline(t, seg)(tn)
    return sps.sosfiltfilt(sps.butter(10, 7, fs=1000, output="sos"), seg, axis=0)

class BolsaMovimiento:
    """Bolsa de "palabras de movimiento": cada flujo se lleva a media 0 y varianza 1, un solo vocabulario de d = 90
    palabras (k-means por minilotes) sobre las muestras de todos los flujos juntos, e histograma por episodio."""
    def __init__(s, cj, ent_idx, k=90, semilla=0):
        from sklearn.cluster import MiniBatchKMeans
        rng = np.random.default_rng(semilla); s.k = k; s.cache = {}
        segs = [g for g in (boshoff_senal(cj, i) for i in rng.choice(ent_idx, min(len(ent_idx), 600), replace=False)) if g is not None]
        pts = np.concatenate([g[rng.choice(len(g), min(len(g), 500), replace=False)] for g in segs])
        s.mu, s.sd = pts.mean(0), pts.std(0) + 1e-9
        s.km = MiniBatchKMeans(k, random_state=semilla, n_init=3).fit(((pts - s.mu) / s.sd).astype(np.float32))
    def __call__(s, cj, i):
        if i not in s.cache:
            g = boshoff_senal(cj, i)
            s.cache[i] = np.full(s.k, np.nan) if g is None else np.bincount(s.km.predict(((g - s.mu) / s.sd).astype(np.float32)), minlength=s.k) / len(g)
        return s.cache[i]

# --- Interfaz común ----------------------------------------------------------------------------------------------
class LineaBase:
    """Ajusta un método previo a un dueño y puntúa episodios (más alto = más dueño; NaN = el método no decide)."""
    def __init__(s, metodo, cj, ent_idx):
        s.metodo, s.cj = metodo, cj; s.cache = {}
        s.bolsa = BolsaMovimiento(cj, ent_idx) if metodo.startswith("boshoff") else None
    def rasgos(s, i):
        if i not in s.cache:
            s.cache[i] = (spu_senal(s.cj, i) if s.metodo == "spu" else
                          rasgos_buriro(s.cj, i, CFG["X_MAX"]) if s.metodo == "buriro" else s.bolsa(s.cj, i))
        return s.cache[i]
    def cubre(s, idx):
        """SPU solo decide en tomas desde una superficie (desde un reposo); en las demás pide la autenticación explícita."""
        if s.metodo != "spu": return np.ones(len(idx), bool)
        return s.cj.meta.origen.to_numpy()[idx] == "SURFACE"
    def ajustar(s, enr, neg):
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.neural_network import MLPClassifier
        from sklearn.svm import OneClassSVM
        from sklearn.preprocessing import StandardScaler
        enr = enr[s.cubre(enr)]
        if s.metodo == "spu":
            senales = [x for x in (s.rasgos(i) for i in enr) if len(x) >= 10]
            s.perfil = spu_perfil(senales) if senales else None; return
        Fe = np.array([s.rasgos(i) for i in enr]); Fe = Fe[np.isfinite(Fe).all(1)]
        if s.metodo == "boshoff_ocsvm":
            s.esc = StandardScaler().fit(Fe); s.clf = OneClassSVM(nu=0.1, gamma="scale").fit(s.esc.transform(Fe)); return
        Fn = np.array([s.rasgos(i) for i in neg]); Fn = Fn[np.isfinite(Fn).all(1)]
        X = np.r_[Fe, Fn]; y = np.r_[np.ones(len(Fe)), np.zeros(len(Fn))]
        if s.metodo == "buriro":
            s.cols = np.argsort(-ganancia_informacion(X, y))[:50]          # 50 rasgos con más ganancia de información
            X = X[:, s.cols]
            s.clf = RandomForestClassifier(100, max_features=int(np.log2(X.shape[1])) + 1, random_state=0).fit(X, y)   # como WEKA
        else:
            s.clf = MLPClassifier((100,), max_iter=500, random_state=0).fit(X, y)
    def puntuar(s, idx):
        out = np.full(len(idx), np.nan)
        if len(idx) == 0: return out
        ok = s.cubre(idx)
        if s.metodo == "spu":
            if s.perfil is None: return out
            for k in np.where(ok)[0]:
                q = s.rasgos(idx[k])
                if len(q) >= 10:
                    out[k] = -_dtw_pond(q, s.perfil, PESOS_SPU, len(q) + len(s.perfil))     # sin banda: DTW completo
            return out
        F_ = np.array([s.rasgos(i) for i in idx]); ok &= np.isfinite(F_).all(1)
        if not ok.any(): return out
        if s.metodo == "buriro": out[ok] = s.clf.predict_proba(F_[ok][:, s.cols])[:, 1]
        elif s.metodo == "boshoff_ocsvm": out[ok] = s.clf.decision_function(s.esc.transform(F_[ok]))
        else: out[ok] = s.clf.predict_proba(F_[ok])[:, 1]
        return out

def correr_linea_base(ID, metodo, grupo="prueba", cj=None, notas="", forzar=False, enrol="N"):
    t = leer_tabla()
    if not forzar and len(t) and ID in set(t.ID): return t[t.ID == ID].iloc[0].to_dict()
    cj = cj or PROPIO; ent, evals = sujetos_de("entrenamiento", cj), sujetos_de(grupo, cj)
    ent_idx = np.concatenate([indices(cj, s) for s in ent])
    neg_idx = np.concatenate([np.random.default_rng(0).choice(indices(cj, s), min(30, len(indices(cj, s))), replace=False) for s in ent])
    lb = LineaBase(metodo, cj, ent_idx); filas = []
    for s in evals:
        div = division_dueno(cj, s, enrol)
        if div is None: continue
        enr, cal, ev = div
        grupos = dict(ev=ev, cal=cal, imp_m=indices(cj, s, impostor=True), imp_o=impostores_otros(cj, s, evals))
        lb.ajustar(enr, neg_idx)
        sc = {k: lb.puntuar(v) for k, v in grupos.items()}
        umbral = np.nanpercentile(sc["cal"], CFG["FRR_OBJETIVO"] * 100) if np.isfinite(sc["cal"]).any() else np.nan
        filas.append(dict(sujeto=s, eer_mismo=eer(sc["ev"], sc["imp_m"]), eer_otros=eer(sc["ev"], sc["imp_o"]),
                          far_mismo=far_en(sc["imp_m"], umbral), far_otros=far_en(sc["imp_o"], umbral),
                          frr=float((sc["ev"][np.isfinite(sc["ev"])] < umbral).mean() * 100) if np.isfinite(sc["ev"]).any() else np.nan,
                          cobertura=float(np.isfinite(sc["ev"]).mean() * 100) if len(ev) else np.nan,
                          puntajes=json.dumps({k: [float(x) for x in v] for k, v in sc.items()})))
    det = pd.DataFrame(filas); det.to_csv(RES / "detalle" / f"{ID}.csv", index=False)
    if det.empty: return {}
    fila = dict(ID=ID, Arquitectura=metodo, Preentrenamiento="—", Objetivo="—", Esquema="línea base", Ajuste="—", Grupo=grupo,
                Enrolamiento=enrol, Ramas={"spu": "toma", "buriro": "uso"}.get(metodo, "toma + 7 s"),
                EER_mismo=det.eer_mismo.mean(), EER_mismo_sd=det.eer_mismo.std(), EER_otros=det.eer_otros.mean(), EER_otros_sd=det.eer_otros.std(),
                FAR_mismo_FRR5=det.far_mismo.mean(), FAR_otros_FRR5=det.far_otros.mean(), FRR_real=det.frr.mean(),
                Cobertura=det.cobertura.mean(),
                Metrica_decision="EER_mismo" if det.eer_mismo.notna().any() else "EER_otros", Personas=len(det), Semillas=1,
                Estado="Hecho", Notas=notas)
    agregar_fila(fila); print(f"{ID} {metodo}: EER otros {fila['EER_otros']:.2f} | EER mismo {fila['EER_mismo']} | cobertura {fila['Cobertura']:.0f} %")
    return fila

## 14. Motion ID: condiciones ideales (camino 2) y campo con el mismo teléfono

**Ideal.** La etapa 2 se entrena sin identidades con las 90 personas de `train_val_test` (sobre las capas bajas de
la etapa 1, como el modelo final). Las 11 de `additional_test` son los dueños: cada una enrola con sus primeras
180 tomas, calibra con las 30 siguientes y se evalúa con las últimas 90, frente a las últimas 90 de las otras 10.
Como en Gavron et al., se reporta el FAR con TAR del 90 % (umbral en el percentil 10 de sus propios puntajes),
remuestreando 90 tomas de impostor cada vez. Su tabla 3 está transcrita abajo para compararla (hay que verificarla
contra el PDF: filas = cantidad de personas del modelo base, columnas = las 11 personas de prueba).

**Campo.** El modelo final (entrenado con el dataset propio) se evalúa en cada par persona-teléfono: impostores del
mismo teléfono = las otras personas cuando usaron ese teléfono. Además, la prueba de fuga persona frente a
teléfono: desde los vectores del episodio se predice el teléfono (entrenando con unas personas y probando con
otras) y la persona (entrenando con unos teléfonos y probando con otros).

In [ ]:
GAVRON_TABLA3 = {   # FAR (%) con TAR = 90 %, por persona de prueba 0–10; transcrito del arXiv:2302.01751v1
    60: [0, 4.2, 9, 12, 2.0, 12, 5, 24, 6, 4, 1.9], 65: [0, 6, 12, 17, 1.0, 14, 3, 23, 8, 2.0, 0.5],
    70: [0, 4, 17, 10, 1.9, 11, 4, 21, 6, 2.4, 0.6], 75: [0.6, 2.0, 10, 12, 5, 11, 2.8, 22, 5, 4, 0.6],
    80: [0, 6, 5, 13, 1.0, 10, 4, 22, 5, 1.6, 0], 85: [1.0, 6, 8, 14, 8, 10, 4.7, 22, 4.9, 2.2, 0.5]}

def far_con_tar(gen, imp, tar=0.90, B=None, n=90, semilla=0):
    gen = np.asarray(gen, float); imp = np.asarray(imp, float); gen, imp = gen[np.isfinite(gen)], imp[np.isfinite(imp)]
    if len(gen) < 5 or len(imp) < 5: return np.nan, np.nan
    umbral = np.percentile(gen, (1 - tar) * 100); rng = np.random.default_rng(semilla); B = B or CFG["BOOTSTRAP"]
    f = [(rng.choice(imp, min(n, len(imp)), replace=False) >= umbral).mean() * 100 for _ in range(B)]
    return float(np.mean(f)), float(np.std(f))

def division_mid(cj, s, n_enr=180, n_cal=30, n_ev=90):
    i = cj.meta.index[cj.meta.sujeto == s].to_numpy(); i = i[np.argsort(cj.meta.intento.to_numpy()[i])]
    if len(i) < n_enr + n_cal + 10:
        n_ev = min(n_ev, len(i) // 3); n_cal = max(3, len(i) // 6); n_enr = len(i) - n_ev - n_cal
    return i[:n_enr], i[n_enr:n_enr + n_cal], i[-n_ev:]

def puntajes_dueno(m0, cab0, cj, enr, grupos, esq, nivel, semilla, x_s=None):
    mu, sd = (t.to(DEV) for t in stats_persona(cj, enr))
    modelos = ajustar_esquemas(m0, cab0, cj, enr, mu, sd, esq, nivel, semilla)
    return puntuar(modelos, cj, enr, grupos, mu, sd, CFG["X_MAX"] if x_s is None else x_s)

def correr_mid_ideal(ID, arq, pre, obj, esquema, nivel, metodo=None, forzar=False):
    t = leer_tabla()
    if not forzar and len(t) and ID in set(t.ID): return t[t.ID == ID].iloc[0].to_dict()
    cj = CJ_MID; ent = sorted(cj.meta.sujeto[cj.meta.grupo == "train_val_test"].unique()); prueba = sorted(cj.meta.sujeto[cj.meta.grupo == "additional_test"].unique())
    filas = []
    sems = CFG["SEMILLAS"] if metodo is None else [0]
    for sem in sems:
        if metodo is None: m, cab = entrenar_e2(arq, pre, obj, sem, cj, ent)
        else:
            ent_idx = np.concatenate([indices(cj, o) for o in ent])
            neg = np.concatenate([np.random.default_rng(0).choice(indices(cj, o), min(30, len(indices(cj, o))), replace=False) for o in ent])
            lb = LineaBase(metodo, cj, ent_idx)
        divs = {s: division_mid(cj, s) for s in prueba}
        for s in prueba:
            enr, cal, ev = divs[s]; imp = np.concatenate([divs[o][2] for o in prueba if o != s])
            if metodo is None:
                sc = puntajes_dueno(m, cab, cj, enr, dict(ev=ev, cal=cal, imp=imp), esquema, nivel, sem)
            else:
                lb.ajustar(enr, neg); sc = dict(ev=lb.puntuar(ev), imp=lb.puntuar(imp))
            far, far_sd = far_con_tar(sc["ev"], sc["imp"], semilla=sem)
            filas.append(dict(sujeto=s, semilla=sem, far_tar90=far, far_tar90_sd=far_sd, eer=eer(sc["ev"], sc["imp"])))
    det = pd.DataFrame(filas); det.to_csv(RES / "detalle" / f"{ID}.csv", index=False)
    por_usuario = det.groupby("sujeto")[["far_tar90", "eer"]].mean()
    g = np.array(list(GAVRON_TABLA3.values())); gav_media = float(g.mean(1).min()), float(g.mean(1).max())
    fila = dict(ID=ID, Arquitectura=metodo or arq, Preentrenamiento=pre if metodo is None else "—", Objetivo=obj if metodo is None else "—",
                Esquema=esquema if metodo is None else "línea base", Ajuste=nivel if metodo is None else "—", Grupo="mid_additional_test",
                EER_mismo=float(por_usuario.eer.mean()), EER_mismo_sd=float(por_usuario.eer.std()),
                FAR_TAR90=float(por_usuario.far_tar90.mean()), FAR_TAR90_sd=float(por_usuario.far_tar90.std()),
                Metrica_decision="FAR con TAR 90 % (Motion ID)", Personas=len(por_usuario), Semillas=len(sems), Estado="Hecho",
                Notas=f"Gavron et al. 2023, tabla 3: FAR medio entre {gav_media[0]:.1f} y {gav_media[1]:.1f} % según la partición (usa negativos de otras personas)")
    agregar_fila(fila); print(f"{ID}: FAR@TAR90 {fila['FAR_TAR90']:.2f} ± {fila['FAR_TAR90_sd']:.2f} | EER {fila['EER_mismo']:.2f} | {fila['Notas']}")
    return fila

def correr_mid_campo(ID, m, cab, esquema, nivel, semilla=0, forzar=False):
    t = leer_tabla()
    if not forzar and len(t) and ID in set(t.ID): return t[t.ID == ID].iloc[0].to_dict()
    cj = CJ_MIDC; meta = cj.meta; filas = []
    pares = meta.groupby(["sujeto", "telefono"]).size()
    for (s, tel), n in pares.items():
        i = np.where((meta.sujeto == s) & (meta.telefono == tel))[0]; i = i[np.argsort(meta.fecha.to_numpy()[i])]
        if len(i) < CFG["ENROLAR_N"] + CFG["CALIBRAR_N"] + 10: continue
        enr, cal, ev = i[:CFG["ENROLAR_N"]], i[CFG["ENROLAR_N"]:CFG["ENROLAR_N"] + CFG["CALIBRAR_N"]], i[CFG["ENROLAR_N"] + CFG["CALIBRAR_N"]:]
        mismo = np.where((meta.sujeto != s) & (meta.telefono == tel))[0]; otros = np.where((meta.sujeto != s) & (meta.telefono != tel))[0]
        rng = np.random.default_rng(0)
        mismo = rng.choice(mismo, min(len(mismo), 300), replace=False) if len(mismo) else mismo
        otros = rng.choice(otros, min(len(otros), 300), replace=False) if len(otros) else otros
        sc = puntajes_dueno(m, cab, cj, enr, dict(ev=ev, cal=cal, mismo=mismo, otros=otros), esquema, nivel, semilla)
        filas.append(dict(sujeto=s, telefono=tel, eer_mismo=eer(sc["ev"], sc["mismo"]), eer_otros=eer(sc["ev"], sc["otros"])))
    det = pd.DataFrame(filas); det.to_csv(RES / "detalle" / f"{ID}.csv", index=False)
    if det.empty: print(ID, "sin pares evaluables"); return {}
    # prueba de fuga: persona frente a teléfono
    Z, ys, yt = [], [], []
    for (s, tel), n in pares.items():
        i = np.where((meta.sujeto == s) & (meta.telefono == tel))[0]
        mu, sd = (q.to(DEV) for q in stats_persona(cj, i))
        Z.append(salidas(m, cab, cj, i, mu, sd, CFG["X_MAX"])["z"].cpu().numpy()); ys += [s] * len(i); yt += [tel] * len(i)
    Z = np.concatenate(Z); ys, yt = np.array(ys), np.array(yt)
    acc_tel, azar_tel = sonda_lineal(Z, yt, ys); acc_per, azar_per = sonda_lineal(Z, ys, yt)
    fila = dict(ID=ID, Arquitectura="modelo final", Esquema=esquema, Ajuste=nivel, Grupo="mid_campo",
                EER_mismo=det.eer_mismo.mean(), EER_mismo_sd=det.eer_mismo.std(), EER_otros=det.eer_otros.mean(), EER_otros_sd=det.eer_otros.std(),
                Brecha=det.eer_mismo.mean() - det.eer_otros.mean(), Metrica_decision="EER_mismo (Motion ID campo)", Personas=len(det), Semillas=1,
                fuga_telefono=acc_tel, fuga_telefono_azar=azar_tel, persona_entre_telefonos=acc_per, persona_azar=azar_per, Estado="Hecho",
                Notas="Fuga: exactitud balanceada al predecir el teléfono (personas no vistas) y la persona (teléfonos no vistos)")
    agregar_fila(fila); print(f"{ID}: EER mismo tel. {fila['EER_mismo']:.2f}, otros {fila['EER_otros']:.2f} | teléfono {acc_tel:.2f} (azar {azar_tel:.2f}) | persona {acc_per:.2f} (azar {azar_per:.2f})")
    return fila

CJ_MID = Conjunto(*MID_E, "mid_esp") if MID_E is not None else None
CJ_MIDC = Conjunto(*MID_C, "mid_campo") if MID_C is not None else None

## 15. Rondas

Cada ronda fija lo mejor de la anterior. R1–R4 deciden con las personas de **validación**; R5, las líneas base y
Motion ID se miden con las de **prueba**. Cada celda se puede volver a ejecutar: las filas ya hechas se leen de la
tabla.

In [ ]:
ORDEN_ESQ = ["conjunto", "svdd", "ae", "pred", "hibrido"]
ORDEN_NIVEL = ["nada", "normalizacion", "cabeza", "adaptadores", "ultimas", "todo"]

def mejor_por(ids, clave):
    t = leer_tabla(); t = t[t.ID.isin(ids)].copy()
    t["dec"] = t.apply(decision, axis=1); t = t[t.dec.notna()]
    if t.empty: raise RuntimeError(f"Sin resultados para {ids}")
    cand = t[t.dec <= t.dec.min() + CFG["UMBRAL_EMPATE"]].copy(); cand["k"] = cand.apply(clave, axis=1)
    return cand.sort_values("k").iloc[0].to_dict()

# --- Ronda 1: arquitectura (Barlow Twins, codificador fijo, sin ajuste) y objetivo de la etapa 1
ids_arq = []
for k, a in enumerate(CFG["ARQUITECTURAS"]):
    correr_fila(f"1.{k + 1}", a, "A", "barlow", "conjunto", "nada", notas="R1 arquitectura"); ids_arq.append(f"1.{k + 1}")
R1A = mejor_por(ids_arq, lambda r: r["Parametros"]); ARQ = R1A["Arquitectura"]
ids_obj = [R1A["ID"]]
for k, o in enumerate(["vicreg", "enmascarado", "entre_sensores"]):
    correr_fila(f"1.{7 + k}", ARQ, "A", o, "conjunto", "nada", notas="R1 objetivo de la etapa 1"); ids_obj.append(f"1.{7 + k}")
R1 = mejor_por(ids_obj, lambda r: CFG["SIMPLICIDAD_OBJ"].index(r["Objetivo"])); OBJ = R1["Objetivo"]
print("Mejor de R1:", ARQ, OBJ, "→", R1["ID"])

In [ ]:
# --- Ronda 2: esquema de aprendizaje del dueño
ids_esq = [R1["ID"]]
for k, e in enumerate(["svdd", "ae", "pred", "hibrido"]):
    correr_fila(f"2.{k + 1}", ARQ, "A", OBJ, e, "cabeza", notas=f"R2; referencia {R1['ID']}"); ids_esq.append(f"2.{k + 1}")
R2 = mejor_por(ids_esq, lambda r: ORDEN_ESQ.index(r["Esquema"])); ESQ = R2["Esquema"]
print("Mejor de R2:", ESQ, "→", R2["ID"])

In [ ]:
# --- Ronda 3: nivel de ajuste al dueño (decide con el enrolamiento estándar; se mide además enrolando con la semana 1 y,
#     de forma provisional mientras no hay 2 semanas de datos, con el día 1 y los días 1–3)
ids_niv = [R2["ID"]]
for k, n in enumerate(["normalizacion", "cabeza", "adaptadores", "ultimas", "todo"]):
    if ESQ != "conjunto" and n == "cabeza" and R2["Ajuste"] == "cabeza": ids_niv.append(R2["ID"]); continue
    correr_fila(f"3.{k + 1}", ARQ, "A", OBJ, ESQ, n, notas="R3"); ids_niv.append(f"3.{k + 1}")
    for sem in CFG["SEMANAS_R3"]:
        correr_fila(f"3.{k + 1}.s{sem}", ARQ, "A", OBJ, ESQ, n, enrol=f"semana{sem}", notas=f"R3, enrolamiento con {sem} semana(s)")
    for d in CFG["DIAS_R3"]:
        correr_fila(f"3.{k + 1}.d{d}", ARQ, "A", OBJ, ESQ, n, enrol=f"dias{d}", notas=f"R3 provisional, enrolamiento con {d} día(s)")
R3 = mejor_por(ids_niv, lambda r: ORDEN_NIVEL.index(r["Ajuste"])); NIVEL = R3["Ajuste"]
print("Mejor de R3:", NIVEL, "→", R3["ID"])

In [ ]:
# --- Ronda 4: controles del preentrenamiento
correr_fila("4.1", ARQ, "B", OBJ, ESQ, NIVEL, notas="Sin datos públicos: etapa 2 desde cero")
correr_fila("4.2", ARQ, "cero", OBJ, ESQ, NIVEL, notas="Desde cero solo con el dueño (sin etapas 1 ni 2; ajuste 'todo')")
correr_fila("4.3", ARQ, "A", OBJ, ESQ, NIVEL, bajas_modo="libre", notas="Etapa 2 sin congelar las capas bajas")

def con_fs(fs):
    """Cambia la frecuencia global (para el control a 100 Hz)."""
    global FS, I_UP
    FS = fs; I_UP = CFG["PRE_S"] * fs

if "4.4" not in set(leer_tabla().ID):
    meta100, X100, V100, F100, B100 = cargar_propio(fs=100)
    meta100["sujeto"] = meta100["sujeto"].astype(str)
    PROPIO100 = Conjunto(meta100, X100, V100, F100, B100, "propio100"); del X100, V100, F100, B100
    con_fs(100)
    try: correr_fila("4.4", ARQ, "B", OBJ, ESQ, NIVEL, cj=PROPIO100, notas="Sin públicos, dataset propio a 100 Hz (comparar con 4.1)")
    finally: con_fs(CFG["FS"]); del PROPIO100

cands = {R3["ID"]: dict(pre="A", bajas=None), "4.1": dict(pre="B", bajas=None), "4.3": dict(pre="A", bajas="libre")}
R4 = mejor_por(list(cands), lambda r: 0 if r["ID"] == R3["ID"] else 1)
PRE, BAJAS = cands[R4["ID"]]["pre"], cands[R4["ID"]]["bajas"]
print("Mejor de R4:", R4["ID"], PRE, BAJAS or CFG["E2_BAJAS"])

In [ ]:
# --- Ronda 5: fases y duración, con el modelo final, en las personas de PRUEBA
final = dict(arq=ARQ, pre=PRE, obj=OBJ, esquema=ESQ, nivel=NIVEL, bajas_modo=BAJAS)
correr_fila("5.0", ARQ, PRE, OBJ, ESQ, NIVEL, bajas_modo=BAJAS, grupo="prueba", notas="Modelo final, todas las ramas; incluye la curva EER frente a x (5.5)")
for ID, ramas, nota in [("5.1", ["toma"], "H2"), ("5.2", ["asentamiento"], "H2"), ("5.3", ["uso"], "H2"), ("5.4", ["toma", "uso"], "H3: sin asentamiento")]:
    correr_fila(ID, ARQ, PRE, OBJ, ESQ, NIVEL, bajas_modo=BAJAS, grupo="prueba", ramas=ramas, notas=nota)

## 16. Comparación con trabajos previos y con Motion ID

In [ ]:
for ID, met, nota in [("B.1", "spu", "Secure Pick Up: DTW ponderado sobre la toma; solo el dueño"),
                      ("B.2", "buriro", "Please Hold On: estadísticos del uso + RF con negativos de otras personas"),
                      ("B.3", "boshoff_mlp", "Boshoff: bolsa de palabras + MLP con negativos de otras personas"),
                      ("B.4", "boshoff_ocsvm", "Boshoff: bolsa de palabras + OC-SVM, solo el dueño")]:
    correr_linea_base(ID, met, grupo="prueba", notas=nota)

if CJ_MID is not None:
    correr_mid_ideal("M.1", ARQ, PRE if PRE != "cero" else "A", OBJ, ESQ, NIVEL)
    for ID, met in [("M.2", "spu"), ("M.3", "buriro"), ("M.4", "boshoff_mlp"), ("M.5", "boshoff_ocsvm")]:
        correr_mid_ideal(ID, None, None, None, None, None, metodo=met)

if CJ_MIDC is not None:
    m_fin, cab_fin = entrenar_e2(ARQ, PRE, OBJ, CFG["SEMILLAS"][0], PROPIO, sujetos_de("entrenamiento"), bajas_modo=BAJAS)
    correr_mid_campo("M.6", m_fin, cab_fin, ESQ, NIVEL)

## 17. Resumen

La tabla completa queda en `tabla_seguimiento.csv`. Abajo, las columnas principales, la curva EER frente a x del
modelo final (H4) y la caída de lo ideal (Motion ID) a lo real (dataset propio) para cada método.

In [ ]:
T = leer_tabla()
cols = ["ID", "Arquitectura", "Preentrenamiento", "Objetivo", "Esquema", "Ajuste", "Grupo", "EER_mismo", "EER_otros",
        "EER_otros_sd", "FAR_otros_FRR5", "FAR_TAR90", "fuga_hhar", "Metrica_decision", "Notas"]
print(T[[c for c in cols if c in T.columns]].to_string(index=False))

f5 = T[T.ID == "5.0"]
if len(f5):
    xs = list(range(0, CFG["X_MAX"] + 1)); y = [f5.iloc[0].get(f"EER_otros@{x}s") for x in xs]; y2 = [f5.iloc[0].get(f"EER_mismo@{x}s") for x in xs]
    fig, ax = plt.subplots(figsize=(6, 4)); ax.plot(xs, y, "o-", label="otros teléfonos")
    if pd.notna(y2).any(): ax.plot(xs, y2, "s-", label="mismo teléfono")
    ax.set_xlabel("segundos de uso después del desbloqueo (x)"); ax.set_ylabel("EER (%)"); ax.legend(); ax.grid(alpha=0.3)
    ax.set_title("H4: EER frente a x (modelo final, personas de prueba)"); fig.tight_layout(); fig.savefig(RES / "eer_vs_x.png", dpi=120)
    plt.show() if EN_JUPYTER else plt.close(fig)

pares = [("modelo final", "5.0", "M.1"), ("Secure Pick Up", "B.1", "M.2"), ("Please Hold On", "B.2", "M.3"),
         ("Boshoff MLP", "B.3", "M.4"), ("Boshoff OC-SVM", "B.4", "M.5")]
filas = []
for nombre, real, ideal in pares:
    r, i = T[T.ID == real], T[T.ID == ideal]
    if len(r) and len(i):
        filas.append(dict(metodo=nombre, EER_ideal_MotionID=i.iloc[0].EER_mismo, EER_real_propio=decision(r.iloc[0]),
                          caida=decision(r.iloc[0]) - i.iloc[0].EER_mismo, FAR_TAR90_MotionID=i.iloc[0].get("FAR_TAR90")))
if filas:
    caida = pd.DataFrame(filas); caida.to_csv(RES / "caida_ideal_a_real.csv", index=False); print(caida.to_string(index=False))
subir_resultados()
print("Listo. Resultados en", RES)